# Unified AIOps Modeling and Operations Notebook

This notebook combines the training/evaluation workflow from `Final (2).ipynb` with the operational pipeline from `aiops_notebook.ipynb`. It covers ticket classification, fallback classification, priority prediction, emotion detection, log relevance detection, semantic retrieval, incident risk scoring, orchestration, and explainable RCA generation.


## Table of Contents

1. Install Dependencies
2. Configuration
3. Data Loading
4. Ticket Classification — DeBERTa-v3-base
5. Classification Fallback — TF-IDF + Logistic Regression
6. Priority Prediction — LightGBM Multiclass
7. Emotion Detection — DistilRoBERTa-base
8. Log Relevance Detection — MiniLM
9. Semantic Retrieval — BGE-base-en-v1.5 + FAISS
10. Incident Risk Prediction — LightGBM Binary
11. Final Inference + Orchestration Layer
12. Rule-Based RCA Generator
13. AIOps Pipeline — Configuration, Celery, Model Loader, Tasks, FastAPI
14. Metrics Summary
15. Manual Pipeline Trigger


## Install Dependencies

Merged dependencies from both source notebooks.


In [ ]:
# Run once in a fresh notebook environment
!pip -q install fastapi==0.111.0 uvicorn[standard]==0.30.1 celery==5.4.0 redis==5.0.4 transformers==4.41.2 torch==2.3.1 sentence-transformers==3.0.1 lightgbm==4.3.0 scikit-learn==1.5.0 numpy==1.26.4 pandas==2.2.2 joblib==1.4.2 openai==1.35.7 requests==2.32.3 azure-storage-file-share==12.16.0 chromadb==0.5.3 python-dotenv==1.0.1 datasets==2.20.0 huggingface_hub==0.23.4 pyarrow gdown kagglehub faiss-cpu


## Configuration

Notebook-local paths and shared imports used by the combined workflow.


In [ ]:
from pathlib import Path
import os
import random
import numpy as np

NOTEBOOK_ROOT = Path.cwd()
NOTEBOOK_BASE = str(NOTEBOOK_ROOT / "jsm_ai_ops_workspace")
DATA_DIR = os.path.join(NOTEBOOK_BASE, "data")
ART = os.path.join(NOTEBOOK_BASE, "artifacts_all")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ART, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import faiss
except Exception:
    faiss = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

print("NOTEBOOK_BASE:", NOTEBOOK_BASE)
print("DATA_DIR:", DATA_DIR)
print("ART:", ART)


## Data Loading

Source: `Final (2).ipynb` cells 0 and 1.


In [ ]:
# CONSOLIDATED COLAB NOTEBOOK (TRAIN + TEST ALL PRIMARY MODELS)
# =============================================================
#
# Required columns (use what applies; notebook handles missing optional columns):
# created_at, summary, description,
# final_category, final_priority, final_emotion,
# ticket_context, log_line, log_relevant,             # for log reranker (pair rows)
# retrieval_query, retrieval_doc, retrieval_group_id, retrieval_relevant,  # for retrieval eval
# incident_in_next_30m,                               # for incident prediction
# comments_text, top_error_lines, affected_users, downtime_minutes, error_count,
# fatal_count, timeout_count, auth_error_count, env, service
# =============================================================

# =============================================================
# PUBLIC DATASET LOADER / NORMALIZER
# Adds public datasets for training/testing and maps them into
# the notebook's expected schema.
# =============================================================


import os
import json
import random
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = NOTEBOOK_BASE
DATA_DIR = f"{BASE}/data"
ART = f"{BASE}/artifacts_all"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ART, exist_ok=True)

PUBLIC_DATA_CSV = f"{DATA_DIR}/tickets_public_combined.csv"

TARGET_COLUMNS = [
    "created_at", "summary", "description",
    "final_category", "final_priority", "final_emotion",
    "ticket_context", "log_line", "log_relevant",
    "retrieval_query", "retrieval_doc", "retrieval_group_id", "retrieval_relevant",
    "incident_in_next_30m",
    "comments_text", "top_error_lines",
    "affected_users", "downtime_minutes", "error_count",
    "fatal_count", "timeout_count", "auth_error_count",
    "env", "service"
]

def empty_frame():
    return pd.DataFrame(columns=TARGET_COLUMNS)

def ensure_schema(df):
    df = df.copy()
    for c in TARGET_COLUMNS:
        if c not in df.columns:
            if c in [
                "log_relevant", "retrieval_relevant", "incident_in_next_30m",
                "affected_users", "downtime_minutes", "error_count",
                "fatal_count", "timeout_count", "auth_error_count"
            ]:
                df[c] = 0
            else:
                df[c] = ""
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    if df["created_at"].isna().all():
        df["created_at"] = pd.Timestamp.utcnow()
    else:
        df["created_at"] = df["created_at"].fillna(pd.Timestamp.utcnow())
    return df[TARGET_COLUMNS]

# -------------------------------------------------------------
# 1) TriageIQ: category + urgency + sentiment
# Maps:
#   category -> final_category
#   urgency  -> final_priority (proxy map)
#   sentiment -> final_emotion
# -------------------------------------------------------------
def load_triageiq(max_rows=None):
    try:
        ds = load_dataset("coldstart88/triageiq-dataset")
        split_name = list(ds.keys())[0]
        pdf = ds[split_name].to_pandas()

        # Defensive column mapping
        cols = {c.lower(): c for c in pdf.columns}
        text_col = cols.get("text") or cols.get("ticket") or cols.get("content")
        cat_col = cols.get("category")
        urg_col = cols.get("urgency")
        emo_col = cols.get("sentiment")

        if text_col is None:
            raise ValueError(f"TriageIQ text column not found. Columns: {list(pdf.columns)}")

        out = empty_frame()
        out["summary"] = pdf[text_col].astype(str).str.slice(0, 160)
        out["description"] = pdf[text_col].astype(str)
        out["comments_text"] = ""
        out["top_error_lines"] = ""
        out["env"] = "unknown"
        out["service"] = "support"

        if cat_col:
            out["final_category"] = pdf[cat_col].astype(str)

        if urg_col:
            urgency_map = {
                "low": "P4",
                "medium": "P3",
                "high": "P2",
                "critical": "P1"
            }
            out["final_priority"] = (
                pdf[urg_col].astype(str).str.lower().map(urgency_map).fillna("P3")
            )

        if emo_col:
            emo_map = {
                "negative": "frustrated",
                "neutral": "neutral",
                "positive": "neutral"
            }
            out["final_emotion"] = (
                pdf[emo_col].astype(str).str.lower().map(emo_map).fillna("neutral")
            )

        out["created_at"] = pd.Timestamp.utcnow()

        if max_rows:
            out = out.sample(min(max_rows, len(out)), random_state=SEED)

        return ensure_schema(out)

    except Exception as e:
        print("Skipping TriageIQ:", e)
        return empty_frame()

# -------------------------------------------------------------
# 2) Support Ticket Intents: augment text/category
# Maps intent-like label -> final_category
# -------------------------------------------------------------
def load_support_ticket_intents(max_rows=None):
    try:
        ds = load_dataset("irongateprd/support-ticket-intents")
        split_name = list(ds.keys())[0]
        pdf = ds[split_name].to_pandas()

        cols = {c.lower(): c for c in pdf.columns}
        text_col = cols.get("text") or cols.get("ticket") or cols.get("utterance") or cols.get("content")
        label_col = cols.get("label") or cols.get("intent") or cols.get("category")

        if text_col is None:
            raise ValueError(f"Support-ticket-intents text column not found. Columns: {list(pdf.columns)}")

        out = empty_frame()
        out["summary"] = pdf[text_col].astype(str).str.slice(0, 160)
        out["description"] = pdf[text_col].astype(str)
        out["comments_text"] = ""
        out["top_error_lines"] = ""
        out["created_at"] = pd.Timestamp.utcnow()
        out["env"] = "unknown"
        out["service"] = "support"

        if label_col:
            out["final_category"] = pdf[label_col].astype(str)

        if max_rows:
            out = out.sample(min(max_rows, len(out)), random_state=SEED)

        return ensure_schema(out)

    except Exception as e:
        print("Skipping support-ticket-intents:", e)
        return empty_frame()

# -------------------------------------------------------------
# 3) BEIR: retrieval eval rows
# Produces:
#   retrieval_query, retrieval_doc, retrieval_group_id, retrieval_relevant
# Uses a small BEIR subset if available.
# -------------------------------------------------------------
def load_beir_subset(dataset_name="msmarco", max_queries=500):
    try:
        # Hugging Face BEIR layout can vary by subset; this block is defensive.
        ds = load_dataset("BeIR/beir", dataset_name)

        available = list(ds.keys())
        print("BEIR splits:", available)

        # Try common split names
        corpus_split = "corpus" if "corpus" in ds else available[0]
        queries_split = "queries" if "queries" in ds else None
        qrels_split = "qrels" if "qrels" in ds else None

        if queries_split is None or qrels_split is None:
            raise ValueError("BEIR subset missing expected queries/qrels splits in this loader.")

        corpus_df = ds[corpus_split].to_pandas()
        queries_df = ds[queries_split].to_pandas()
        qrels_df = ds[qrels_split].to_pandas()

        # Normalize likely field names
        corpus_cols = {c.lower(): c for c in corpus_df.columns}
        queries_cols = {c.lower(): c for c in queries_df.columns}
        qrels_cols = {c.lower(): c for c in qrels_df.columns}

        doc_id_col = corpus_cols.get("_id") or corpus_cols.get("id") or corpus_cols.get("doc_id")
        text_col = corpus_cols.get("text") or corpus_cols.get("contents")
        title_col = corpus_cols.get("title")

        qid_col = queries_cols.get("_id") or queries_cols.get("id") or queries_cols.get("query_id")
        query_col = queries_cols.get("text") or queries_cols.get("query")

        qrels_qid = qrels_cols.get("query-id") or qrels_cols.get("query_id") or qrels_cols.get("qid")
        qrels_docid = qrels_cols.get("corpus-id") or qrels_cols.get("corpus_id") or qrels_cols.get("doc_id")
        qrels_score = qrels_cols.get("score") or qrels_cols.get("relevance")

        if not all([doc_id_col, text_col, qid_col, query_col, qrels_qid, qrels_docid]):
            raise ValueError("Could not identify BEIR columns.")

        if title_col:
            corpus_df["joined_doc"] = corpus_df[title_col].fillna("").astype(str) + " " + corpus_df[text_col].fillna("").astype(str)
        else:
            corpus_df["joined_doc"] = corpus_df[text_col].fillna("").astype(str)

        corpus_map = corpus_df[[doc_id_col, "joined_doc"]].drop_duplicates()
        query_map = queries_df[[qid_col, query_col]].drop_duplicates()

        merged = qrels_df.merge(query_map, left_on=qrels_qid, right_on=qid_col, how="inner")
        merged = merged.merge(corpus_map, left_on=qrels_docid, right_on=doc_id_col, how="inner")

        if qrels_score:
            merged["label"] = (pd.to_numeric(merged[qrels_score], errors="coerce").fillna(0) > 0).astype(int)
        else:
            merged["label"] = 1

        merged = merged.rename(columns={
            query_col: "retrieval_query",
            "joined_doc": "retrieval_doc",
            qrels_qid: "retrieval_group_id"
        })

        merged = merged[["retrieval_query", "retrieval_doc", "retrieval_group_id", "label"]].copy()
        merged["retrieval_relevant"] = merged["label"]
        merged.drop(columns=["label"], inplace=True)

        if max_queries:
            keep_q = merged["retrieval_group_id"].astype(str).drop_duplicates().head(max_queries)
            merged = merged[merged["retrieval_group_id"].astype(str).isin(set(keep_q))]

        out = empty_frame()
        out["created_at"] = pd.Timestamp.utcnow()
        out["retrieval_query"] = merged["retrieval_query"].astype(str)
        out["retrieval_doc"] = merged["retrieval_doc"].astype(str)
        out["retrieval_group_id"] = merged["retrieval_group_id"].astype(str)
        out["retrieval_relevant"] = merged["retrieval_relevant"].astype(int)
        out["service"] = "retrieval"
        out["env"] = "benchmark"

        return ensure_schema(out)

    except Exception as e:
        print("Skipping BEIR subset:", e)
        return empty_frame()

# -------------------------------------------------------------
# 4) Optional local CSV merge
# Keeps your existing tickets.csv if present
# -------------------------------------------------------------
def load_local_csv(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return empty_frame()
    try:
        pdf = pd.read_csv(path)
    except Exception:
        try:
            pdf = pd.read_csv(path, sep=";")
        except Exception:
            pdf = pd.read_csv(path, sep="\t")
    return ensure_schema(pdf)

# -------------------------------------------------------------
# Build combined dataset
# -------------------------------------------------------------
LOCAL_CSV = f"{DATA_DIR}/tickets.csv"

parts = [
    load_local_csv(LOCAL_CSV),
    load_triageiq(max_rows=5000),
    load_support_ticket_intents(max_rows=5000),
    load_beir_subset(dataset_name="msmarco", max_queries=300),
]

df = pd.concat(parts, ignore_index=True)
df = ensure_schema(df)
df = df.drop_duplicates(subset=[
    "summary", "description", "final_category", "final_priority",
    "final_emotion", "retrieval_query", "retrieval_doc", "log_line"
]).reset_index(drop=True)

df.to_csv(PUBLIC_DATA_CSV, index=False)

print("Combined dataset shape:", df.shape)
print("Saved:", PUBLIC_DATA_CSV)
print("Columns:", df.columns.tolist())

print("\nNon-empty label coverage:")
for c in ["final_category", "final_priority", "final_emotion", "retrieval_query", "log_line", "incident_in_next_30m"]:
    if c in df.columns:
        if df[c].dtype == object:
            n = int((df[c].astype(str).str.len() > 0).sum())
        else:
            n = int(df[c].notna().sum())
        print(f"{c}: {n}")
# ----------------------------
# Basic cleanup
# ----------------------------
def ensure_col(col, default):
    if col not in df.columns:
        df[col] = default
    df[col] = df[col].fillna(default)

for c in ["summary","description","comments_text","top_error_lines","env","service",
          "final_category","final_priority","final_emotion",
          "ticket_context","log_line","retrieval_query","retrieval_doc","retrieval_group_id"]:
    ensure_col(c, "")

for c in ["affected_users","downtime_minutes","error_count","fatal_count","timeout_count","auth_error_count",
          "log_relevant","retrieval_relevant","incident_in_next_30m"]:
    if c not in df.columns:
        df[c] = 0
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["created_at"] = pd.to_datetime(df.get("created_at", pd.Timestamp.utcnow()), errors="coerce")
df = df.dropna(subset=["created_at"]).sort_values("created_at").reset_index(drop=True)

# Strict priority filtering
ALLOWED_P = {"P1","P2","P3","P4","P5"}
df["final_priority"] = df["final_priority"].astype(str).str.upper().str.strip()
df = df[(df["final_priority"] == "") | (df["final_priority"].isin(ALLOWED_P))].copy()

def time_split(frame, train=0.70, val=0.15):
    n = len(frame)
    i1, i2 = int(n*train), int(n*(train+val))
    return frame.iloc[:i1].copy(), frame.iloc[i1:i2].copy(), frame.iloc[i2:].copy()

train_df, val_df, test_df = time_split(df)

print("Total:", len(df), "Train/Val/Test:", len(train_df), len(val_df), len(test_df))

summary_metrics = {}


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import torch

from datasets import Dataset


from sklearn.metrics import f1_score, accuracy_score, average_precision_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


## Section 1: Ticket Classification — DeBERTa-v3-base

Primary transformer classifier with built-in fallback handling.


In [ ]:
# # ============================================================
# # 1) CATEGORY MODEL — DeBERTa-v3-base (with fallback)
# # Compatible with newer transformers Trainer API
# # ============================================================


# cat_df = df[df["final_category"].astype(str).str.len() > 0].copy()

# MIN_DEEP_TRAIN_ROWS = 200
# MIN_LIGHT_TRAIN_ROWS = 30   # fallback threshold

# def mk_text(x):
#     return (
#         str(x.get("summary", "")) + " [SEP] " +
#         str(x.get("description", "")) + " [SEP] " +
#         str(x.get("comments_text", "")) + " [SEP] " +
#         str(x.get("top_error_lines", ""))
#     )

# def build_trainer(model, args, train_dataset, eval_dataset, tok, compute_metrics):
#     return Trainer(
#         model=model,
#         args=args,
#         train_dataset=train_dataset,
#         eval_dataset=eval_dataset,
#         data_collator=DataCollatorWithPadding(tokenizer=tok),
#         compute_metrics=compute_metrics
#     )

# if len(cat_df) >= MIN_DEEP_TRAIN_ROWS:
#     tr, va, te = time_split(cat_df)

#     tr["text"] = tr.apply(mk_text, axis=1)
#     va["text"] = va.apply(mk_text, axis=1)
#     te["text"] = te.apply(mk_text, axis=1)

#     le_cat = LabelEncoder()
#     y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
#     y_va = le_cat.transform(va["final_category"].astype(str))
#     y_te = le_cat.transform(te["final_category"].astype(str))

#     ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": y_tr}))
#     ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": y_va}))
#     ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": y_te}))

#     model_name = "microsoft/deberta-v3-base"
#     tok = AutoTokenizer.from_pretrained(model_name)

#     def tok_fn(b):
#         return tok(b["text"], truncation=True, max_length=384)

#     ds_tr = ds_tr.map(tok_fn, batched=True)
#     ds_va = ds_va.map(tok_fn, batched=True)
#     ds_te = ds_te.map(tok_fn, batched=True)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name,
#         num_labels=len(le_cat.classes_)
#     )

#     def comp(eval_pred):
#         logits, labels = eval_pred
#         pred = np.argmax(logits, axis=-1)
#         return {
#             "macro_f1": f1_score(labels, pred, average="macro"),
#             "accuracy": accuracy_score(labels, pred)
#         }

#     out_dir = f"{ART}/classification_deberta"
#     args = TrainingArguments(
#         output_dir=out_dir,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         learning_rate=2e-5,
#         per_device_train_batch_size=8,
#         per_device_eval_batch_size=16,
#         num_train_epochs=3,
#         weight_decay=0.01,
#         warmup_steps=10,
#         fp16=torch.cuda.is_available(),
#         load_best_model_at_end=True,
#         metric_for_best_model="macro_f1",
#         save_total_limit=2,
#         report_to="none"
#     )

#     trainer = build_trainer(
#         model=model,
#         args=args,
#         train_dataset=ds_tr,
#         eval_dataset=ds_va,
#         tok=tok,
#         compute_metrics=comp
#     )

#     trainer.train()
#     pred = trainer.predict(ds_te)
#     yhat = np.argmax(pred.predictions, axis=-1)
#     m = f1_score(y_te, yhat, average="macro")
#     summary_metrics["classification_macro_f1"] = float(m)

#     trainer.save_model(out_dir)
#     tok.save_pretrained(out_dir)
#     joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
#     print("Classification Macro-F1:", round(m, 4))

# elif len(cat_df) >= MIN_LIGHT_TRAIN_ROWS:
#     from sklearn.pipeline import Pipeline
#     from sklearn.feature_extraction.text import TfidfVectorizer
#     from sklearn.linear_model import LogisticRegression

#     tr, va, te = time_split(cat_df)

#     tr["text"] = tr.apply(mk_text, axis=1)
#     va["text"] = va.apply(mk_text, axis=1)
#     te["text"] = te.apply(mk_text, axis=1)

#     le_cat = LabelEncoder()
#     y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
#     y_te = le_cat.transform(te["final_category"].astype(str))

#     clf = Pipeline([
#         ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
#         ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))
#     ])

#     clf.fit(tr["text"], y_tr)
#     yhat = clf.predict(te["text"])
#     m = f1_score(y_te, yhat, average="macro")
#     summary_metrics["classification_macro_f1"] = float(m)

#     out_dir = f"{ART}/classification_tfidf_lr"
#     os.makedirs(out_dir, exist_ok=True)
#     joblib.dump(clf, f"{out_dir}/model.joblib")
#     joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
#     print("Fallback Classification (TF-IDF+LR) Macro-F1:", round(m, 4))

# else:
#     summary_metrics["classification_macro_f1"] = None
#     print(
#         f"Skipped supervised classification (labeled rows={len(cat_df)}). "
#         f"Use rule-based/zero-shot inference fallback at runtime."
#     )

# ============================================================
# 1) CATEGORY MODEL — DeBERTa-v3-base (with fallback)
# Updated for newer transformers + XLA-safe settings
# ============================================================

# Make sure these imports were run earlier:
# from datasets import Dataset
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, accuracy_score
# from transformers import (
#     AutoTokenizer, AutoModelForSequenceClassification,
#     TrainingArguments, Trainer, DataCollatorWithPadding
# )

cat_df = df[df["final_category"].astype(str).str.len() > 0].copy()

MIN_DEEP_TRAIN_ROWS = 200
MIN_LIGHT_TRAIN_ROWS = 30   # fallback threshold

def mk_text(x):
    return (
        str(x.get("summary", "")) + " [SEP] " +
        str(x.get("description", "")) + " [SEP] " +
        str(x.get("comments_text", "")) + " [SEP] " +
        str(x.get("top_error_lines", ""))
    )

def is_xla_runtime():
    try:
        import torch_xla  # noqa: F401
        return True
    except Exception:
        return False

USE_FP16 = torch.cuda.is_available() and not is_xla_runtime()
USE_BF16 = False

def build_trainer(model, args, train_dataset, eval_dataset, tok, compute_metrics):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=compute_metrics
    )

def build_args(out_dir, train_bs=8, eval_bs=16, epochs=3, lr=2e-5, metric="macro_f1"):
    return TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=10,
        fp16=USE_FP16,
        bf16=USE_BF16,
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model=metric,
        save_total_limit=2,
        report_to="none"
    )

print("Category rows:", len(cat_df), "| CUDA:", torch.cuda.is_available(), "| XLA:", is_xla_runtime(), "| fp16:", USE_FP16)

if len(cat_df) >= MIN_DEEP_TRAIN_ROWS:
    tr, va, te = time_split(cat_df)

    tr["text"] = tr.apply(mk_text, axis=1)
    va["text"] = va.apply(mk_text, axis=1)
    te["text"] = te.apply(mk_text, axis=1)

    le_cat = LabelEncoder()
    y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
    y_va = le_cat.transform(va["final_category"].astype(str))
    y_te = le_cat.transform(te["final_category"].astype(str))

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": y_tr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": y_va}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": y_te}))

    model_name = "microsoft/deberta-v3-base"
    tok = AutoTokenizer.from_pretrained(model_name)

    def tok_fn(batch):
        return tok(batch["text"], truncation=True, max_length=384)

    ds_tr = ds_tr.map(tok_fn, batched=True)
    ds_va = ds_va.map(tok_fn, batched=True)
    ds_te = ds_te.map(tok_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(le_cat.classes_)
    )

    def comp(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/classification_deberta"
    args = build_args(
        out_dir=out_dir,
        train_bs=8,
        eval_bs=16,
        epochs=3,
        lr=2e-5,
        metric="macro_f1"
    )

    trainer = build_trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        tok=tok,
        compute_metrics=comp
    )

    trainer.train()
    pred = trainer.predict(ds_te)
    yhat = np.argmax(pred.predictions, axis=-1)
    m = f1_score(y_te, yhat, average="macro")
    summary_metrics["classification_macro_f1"] = float(m)

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
    print("Classification Macro-F1:", round(m, 4))

elif len(cat_df) >= MIN_LIGHT_TRAIN_ROWS:
    # -------------------------------
    # Fallback: TF-IDF + LogisticRegression
    # -------------------------------
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    tr, va, te = time_split(cat_df)

    tr["text"] = tr.apply(mk_text, axis=1)
    va["text"] = va.apply(mk_text, axis=1)
    te["text"] = te.apply(mk_text, axis=1)

    le_cat = LabelEncoder()
    y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
    y_te = le_cat.transform(te["final_category"].astype(str))

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ])

    clf.fit(tr["text"], y_tr)
    yhat = clf.predict(te["text"])
    m = f1_score(y_te, yhat, average="macro")
    summary_metrics["classification_macro_f1"] = float(m)

    out_dir = f"{ART}/classification_tfidf_lr"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(clf, f"{out_dir}/model.joblib")
    joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
    print("Fallback Classification (TF-IDF+LR) Macro-F1:", round(m, 4))

else:
    # -------------------------------
    # Tiny-data mode: no training
    # -------------------------------
    summary_metrics["classification_macro_f1"] = None
    print(
        f"Skipped supervised classification (labeled rows={len(cat_df)}). "
        f"Use rule-based/zero-shot inference fallback at runtime."
    )

## Section 2: Classification Fallback — TF-IDF + Logistic Regression

Explicit standalone view of the lightweight fallback path.


In [ ]:
# Standalone lightweight fallback for ticket classification
# The main DeBERTa section above already contains automatic branch selection.
# This cell exposes the TF-IDF + Logistic Regression fallback explicitly.

from sklearn.linear_model import LogisticRegression


def train_classification_fallback_tfidf_lr(category_frame, output_dir=ART):
    working = category_frame.copy()
    tr, va, te = time_split(working)

    for frame in (tr, va, te):
        frame["text"] = frame.apply(mk_text, axis=1)

    le_cat = LabelEncoder()
    y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
    y_te = le_cat.transform(te["final_category"].astype(str))

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])

    clf.fit(tr["text"], y_tr)
    yhat = clf.predict(te["text"])
    macro = f1_score(y_te, yhat, average="macro")
    summary_metrics["classification_fallback_macro_f1"] = float(macro)

    out_dir = f"{output_dir}/classification_tfidf_lr"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(clf, f"{out_dir}/model.joblib")
    joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")

    return {"macro_f1": float(macro), "out_dir": out_dir}


if len(cat_df) < MIN_DEEP_TRAIN_ROWS and len(cat_df) >= MIN_LIGHT_TRAIN_ROWS:
    fallback_result = train_classification_fallback_tfidf_lr(cat_df)
    print("Fallback Classification (TF-IDF+LR) Macro-F1:", round(fallback_result["macro_f1"], 4))
else:
    print(
        "Fallback training cell is ready. "
        "The category section above already applies this branch automatically when needed."
    )


## Section 3: Priority Prediction — LightGBM Multiclass


In [ ]:
# ============================================================
# 2) PRIORITY MODEL — LightGBM multiclass P1..P5
# ============================================================
prio_df = df[df["final_priority"].isin(list(ALLOWED_P))].copy()
if len(prio_df) > 300:
    tr, va, te = time_split(prio_df)

    for frame in [tr,va,te]:
        frame["text_merged"] = (
            frame["summary"].astype(str) + " " +
            frame["description"].astype(str) + " " +
            frame["comments_text"].astype(str) + " " +
            frame["top_error_lines"].astype(str)
        )

    num_cols = ["affected_users","downtime_minutes","error_count","fatal_count","timeout_count","auth_error_count"]
    cat_cols = ["env","service"]

    le_p = LabelEncoder()
    ytr = le_p.fit_transform(tr["final_priority"])
    yte = le_p.transform(te["final_priority"])

    pre = ColumnTransformer([
        ("txt", TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=3), "text_merged"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ])

    clf = lgb.LGBMClassifier(
        objective="multiclass", num_class=5,
        learning_rate=0.05, num_leaves=63,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
        min_data_in_leaf=50, reg_alpha=1.0, reg_lambda=2.0,
        n_estimators=700, random_state=42
    )

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(tr[["text_merged"]+cat_cols+num_cols], ytr)
    pred = pipe.predict(te[["text_merged"]+cat_cols+num_cols])

    macro = f1_score(yte, pred, average="macro")
    canon = ["P1","P2","P3","P4","P5"]
    ord_map = {p:i for i,p in enumerate(canon)}
    true_ord = te["final_priority"].map(ord_map).values
    pred_lbl = le_p.inverse_transform(pred)
    pred_ord = pd.Series(pred_lbl).map(ord_map).values
    off1 = float(np.mean(np.abs(true_ord - pred_ord) <= 1))

    summary_metrics["priority_macro_f1"] = float(macro)
    summary_metrics["priority_off_by_1"] = off1

    out_dir = f"{ART}/priority_lgbm"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(pipe, f"{out_dir}/model.joblib")
    joblib.dump(le_p, f"{out_dir}/label_encoder.joblib")
    print("Priority Macro-F1:", round(macro,4), "| Off-by-1:", round(off1,4))
else:
    print("Skipped priority (insufficient labeled rows).")

## Section 4: Emotion Detection — DistilRoBERTa-base


In [ ]:
# # ============================================================
# # 3) EMOTION MODEL — DistilRoBERTa
# # ============================================================
# emo_labels = {"angry","frustrated","urgent","neutral"}
# emo_df = df[df["final_emotion"].astype(str).str.lower().isin(emo_labels)].copy()
# if len(emo_df) > 200:
#     emo_df["final_emotion"] = emo_df["final_emotion"].str.lower()
#     tr, va, te = time_split(emo_df)

#     def mk_text2(x):
#         return x["summary"] + " [SEP] " + x["description"] + " [SEP] " + x["comments_text"]

#     tr["text"] = tr.apply(mk_text2, axis=1)
#     va["text"] = va.apply(mk_text2, axis=1)
#     te["text"] = te.apply(mk_text2, axis=1)

#     le_e = LabelEncoder()
#     ytr = le_e.fit_transform(tr["final_emotion"])
#     yva = le_e.transform(va["final_emotion"])
#     yte = le_e.transform(te["final_emotion"])

#     ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
#     ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
#     ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

#     mname = "distilroberta-base"
#     tok = AutoTokenizer.from_pretrained(mname)
#     def tf(b): return tok(b["text"], truncation=True, max_length=256)
#     ds_tr = ds_tr.map(tf, batched=True)
#     ds_va = ds_va.map(tf, batched=True)
#     ds_te = ds_te.map(tf, batched=True)

#     model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=len(le_e.classes_))

#     def comp2(ep):
#         logits, labels = ep
#         p = np.argmax(logits, axis=-1)
#         return {"macro_f1": f1_score(labels, p, average="macro"), "accuracy": accuracy_score(labels,p)}

#     out_dir = f"{ART}/emotion_distilroberta"
#     args = TrainingArguments(
#         output_dir=out_dir,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         learning_rate=3e-5,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=32,
#         num_train_epochs=4,
#         warmup_ratio=0.06,
#         weight_decay=0.01,
#         fp16=torch.cuda.is_available(),
#         load_best_model_at_end=True,
#         metric_for_best_model="macro_f1",
#         save_total_limit=2,
#         report_to="none"
#     )
#     trainer = Trainer(
#         model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
#         tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=comp2
#     )
#     trainer.train()
#     pred = trainer.predict(ds_te)
#     yhat = np.argmax(pred.predictions, axis=-1)
#     macro = f1_score(yte, yhat, average="macro")
#     summary_metrics["emotion_macro_f1"] = float(macro)

#     trainer.save_model(out_dir); tok.save_pretrained(out_dir); joblib.dump(le_e, f"{out_dir}/label_encoder.joblib")
#     print("Emotion Macro-F1:", round(macro,4))
# else:
#     print("Skipped emotion (insufficient labeled rows).")

# ============================================================
# 3) EMOTION MODEL — DistilRoBERTa (updated)
# ============================================================

emo_labels = {"angry", "frustrated", "urgent", "neutral"}
emo_df = df[df["final_emotion"].astype(str).str.lower().isin(emo_labels)].copy()

# Reuse helpers if already defined above; otherwise define them here.
def is_xla_runtime():
    try:
        import torch_xla  # noqa: F401
        return True
    except Exception:
        return False

USE_FP16 = torch.cuda.is_available() and not is_xla_runtime()
USE_BF16 = False

def build_trainer(model, args, train_dataset, eval_dataset, tok, compute_metrics):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=compute_metrics
    )

def build_args(out_dir, train_bs=16, eval_bs=32, epochs=4, lr=3e-5, metric="macro_f1"):
    return TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        warmup_steps=10,
        weight_decay=0.01,
        fp16=USE_FP16,
        bf16=USE_BF16,
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model=metric,
        save_total_limit=2,
        report_to="none"
    )

if len(emo_df) > 200:
    emo_df["final_emotion"] = emo_df["final_emotion"].astype(str).str.lower()
    tr, va, te = time_split(emo_df)

    def mk_text2(x):
        return (
            str(x.get("summary", "")) + " [SEP] " +
            str(x.get("description", "")) + " [SEP] " +
            str(x.get("comments_text", ""))
        )

    tr["text"] = tr.apply(mk_text2, axis=1)
    va["text"] = va.apply(mk_text2, axis=1)
    te["text"] = te.apply(mk_text2, axis=1)

    le_e = LabelEncoder()
    ytr = le_e.fit_transform(tr["final_emotion"])
    yva = le_e.transform(va["final_emotion"])
    yte = le_e.transform(te["final_emotion"])

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

    mname = "distilroberta-base"
    tok = AutoTokenizer.from_pretrained(mname)

    def tf(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds_tr = ds_tr.map(tf, batched=True)
    ds_va = ds_va.map(tf, batched=True)
    ds_te = ds_te.map(tf, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        mname,
        num_labels=len(le_e.classes_)
    )

    def comp2(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/emotion_distilroberta"
    args = build_args(
        out_dir=out_dir,
        train_bs=16,
        eval_bs=32,
        epochs=4,
        lr=3e-5,
        metric="macro_f1"
    )

    trainer = build_trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        tok=tok,
        compute_metrics=comp2
    )

    trainer.train()
    pred = trainer.predict(ds_te)
    yhat = np.argmax(pred.predictions, axis=-1)
    macro = f1_score(yte, yhat, average="macro")
    summary_metrics["emotion_macro_f1"] = float(macro)

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    joblib.dump(le_e, f"{out_dir}/label_encoder.joblib")

    print("Emotion Macro-F1:", round(macro, 4))
else:
    print("Skipped emotion (insufficient labeled rows).")

## Section 5: Log Relevance Detection — MiniLM (all-MiniLM-L6-v2)


In [ ]:
# # ============================================================
# # 4) LOG RELEVANCE RERANKER — MiniLM pair classifier
# # ============================================================
# # Needs row-level pairs: ticket_context, log_line, log_relevant (0/1)
# log_df = df[(df["ticket_context"].str.len()>0) & (df["log_line"].str.len()>0)].copy()
# if len(log_df) > 500 and log_df["log_relevant"].nunique() > 1:
#     tr, va, te = time_split(log_df)

#     def pair_text(x):
#         return x["ticket_context"] + " [SEP] " + x["log_line"]

#     tr["text"] = tr.apply(pair_text, axis=1)
#     va["text"] = va.apply(pair_text, axis=1)
#     te["text"] = te.apply(pair_text, axis=1)

#     ytr = tr["log_relevant"].astype(int).values
#     yva = va["log_relevant"].astype(int).values
#     yte = te["log_relevant"].astype(int).values

#     ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
#     ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
#     ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

#     mname = "sentence-transformers/all-MiniLM-L6-v2"
#     tok = AutoTokenizer.from_pretrained(mname)
#     model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)

#     def tf(b): return tok(b["text"], truncation=True, max_length=256)
#     ds_tr = ds_tr.map(tf, batched=True); ds_va = ds_va.map(tf, batched=True); ds_te = ds_te.map(tf, batched=True)

#     def comp3(ep):
#         logits, labels = ep
#         pred = np.argmax(logits, axis=-1)
#         return {"macro_f1": f1_score(labels, pred, average="macro"), "accuracy": accuracy_score(labels,pred)}

#     out_dir = f"{ART}/log_reranker_minilm"
#     args = TrainingArguments(
#         output_dir=out_dir, eval_strategy="epoch", save_strategy="epoch",
#         learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
#         num_train_epochs=3, fp16=torch.cuda.is_available(), load_best_model_at_end=True,
#         metric_for_best_model="macro_f1", report_to="none"
#     )
#     trainer = Trainer(
#         model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
#         tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=comp3
#     )
#     trainer.train()

#     pred = trainer.predict(ds_te)
#     logits = pred.predictions
#     probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()

#     # Precision@20 approximation (global)
#     idx = np.argsort(-probs)[:20]
#     p20 = float(np.mean(yte[idx])) if len(idx)>0 else 0.0
#     yhat = (probs >= 0.5).astype(int)
#     macro = f1_score(yte, yhat, average="macro")

#     summary_metrics["log_rerank_macro_f1"] = float(macro)
#     summary_metrics["log_rerank_precision_at_20"] = p20

#     trainer.save_model(out_dir); tok.save_pretrained(out_dir)
#     print("Log Reranker Macro-F1:", round(macro,4), "| P@20:", round(p20,4))
# else:
#     print("Skipped log reranker (need ticket_context, log_line, log_relevant with enough rows).")

# ============================================================
# 4) LOG RELEVANCE RERANKER — MiniLM pair classifier
# With fallback to rule-based and weak supervision
# ============================================================

print("\n=== LOG RELEVANCE RERANKER ===")

log_df = df[(df["ticket_context"].astype(str).str.len() > 0) &
            (df["log_line"].astype(str).str.len() > 0)].copy()

print(f"Paired rows (ticket_context + log_line): {len(log_df)}")
print(f"log_relevant unique values: {log_df['log_relevant'].nunique() if len(log_df) > 0 else 0}")

if len(log_df) > 0:
    print(f"log_relevant distribution:\n{log_df['log_relevant'].value_counts(dropna=False)}")

# ----------------------------
# Option 1: Full supervised training (>500 pairs with both labels)
# ----------------------------
if len(log_df) > 500 and log_df["log_relevant"].nunique() > 1:
    print("\n✓ Sufficient labeled pairs. Training MiniLM reranker...")

    tr, va, te = time_split(log_df)

    def pair_text(x):
        return str(x["ticket_context"]) + " [SEP] " + str(x["log_line"])

    tr["text"] = tr.apply(pair_text, axis=1)
    va["text"] = va.apply(pair_text, axis=1)
    te["text"] = te.apply(pair_text, axis=1)

    ytr = tr["log_relevant"].astype(int).values
    yva = va["log_relevant"].astype(int).values
    yte = te["log_relevant"].astype(int).values

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

    mname = "sentence-transformers/all-MiniLM-L6-v2"
    tok = AutoTokenizer.from_pretrained(mname)
    model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)

    def tf(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds_tr = ds_tr.map(tf, batched=True)
    ds_va = ds_va.map(tf, batched=True)
    ds_te = ds_te.map(tf, batched=True)

    def comp3(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/log_reranker_minilm"
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=3,
        warmup_steps=5,
        fp16=torch.cuda.is_available() and not is_xla_runtime(),
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="none",
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=comp3
    )

    trainer.train()

    pred = trainer.predict(ds_te)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()

    # Precision@20 approximation
    idx = np.argsort(-probs)[:20]
    p20 = float(np.mean(yte[idx])) if len(idx) > 0 else 0.0
    yhat = (probs >= 0.5).astype(int)
    macro = f1_score(yte, yhat, average="macro")

    summary_metrics["log_rerank_macro_f1"] = float(macro)
    summary_metrics["log_rerank_precision_at_20"] = p20
    summary_metrics["log_rerank_mode"] = "supervised_minilm"

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    print(f"✓ Log Reranker Macro-F1: {round(macro, 4)} | P@20: {round(p20, 4)}")

# ----------------------------
# Option 2: Weak supervision (30-500 pairs or imbalanced labels)
# ----------------------------
elif len(log_df) > 30 and log_df["log_relevant"].nunique() > 1:
    print("\n⚠ Limited labeled pairs. Training with weak supervision...")

    tr, va, te = time_split(log_df)

    def pair_text(x):
        return str(x["ticket_context"]) + " [SEP] " + str(x["log_line"])

    tr["text"] = tr.apply(pair_text, axis=1)
    va["text"] = va.apply(pair_text, axis=1)
    te["text"] = te.apply(pair_text, axis=1)

    ytr = tr["log_relevant"].astype(int).values
    yva = va["log_relevant"].astype(int).values
    yte = te["log_relevant"].astype(int).values

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

    mname = "sentence-transformers/all-MiniLM-L6-v2"
    tok = AutoTokenizer.from_pretrained(mname)
    model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)

    def tf(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds_tr = ds_tr.map(tf, batched=True)
    ds_va = ds_va.map(tf, batched=True)
    ds_te = ds_te.map(tf, batched=True)

    def comp3(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/log_reranker_minilm_weak"
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        warmup_steps=2,
        weight_decay=0.01,
        fp16=torch.cuda.is_available() and not is_xla_runtime(),
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="none",
        logging_steps=10
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=comp3
    )

    trainer.train()

    pred = trainer.predict(ds_te)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()

    idx = np.argsort(-probs)[:min(20, len(probs))]
    p20 = float(np.mean(yte[idx])) if len(idx) > 0 else 0.0
    yhat = (probs >= 0.5).astype(int)
    macro = f1_score(yte, yhat, average="macro", zero_division=0)

    summary_metrics["log_rerank_macro_f1"] = float(macro)
    summary_metrics["log_rerank_precision_at_20"] = p20
    summary_metrics["log_rerank_mode"] = "weak_supervision_minilm"

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    print(f"⚠ Log Reranker (weak) Macro-F1: {round(macro, 4)} | P@20: {round(p20, 4)}")

# ----------------------------
# Option 3: Rule-based fallback (no sufficient labels)
# ----------------------------
else:
    print(f"\n✗ Insufficient labeled pairs ({len(log_df)} rows with <2 label classes).")
    print("  Falling back to rule-based log reranker...")

    def rule_based_log_relevance(ticket_context, log_line):
        """
        Simple heuristic relevance score based on keyword/token overlap
        and error signal detection.
        """
        context = str(ticket_context).lower()
        log = str(log_line).lower()

        score = 0

        # Exact token overlap
        context_tokens = set(context.split())
        log_tokens = set(log.split())
        overlap = len(context_tokens & log_tokens)
        score += overlap

        # Error/exception/failure signals
        error_signals = ["error", "exception", "failed", "failure", "fault", "crash"]
        for signal in error_signals:
            if signal in log:
                score += 2

        # Timeout signals
        if "timeout" in log or "timed out" in log:
            score += 2

        # Authentication/authorization signals
        if "auth" in log or "unauthorized" in log or "forbidden" in log:
            score += 2

        # Stack trace or traceback
        if "traceback" in log or "stack" in log:
            score += 1

        return float(score)

    # Save rule-based reranker config
    out_dir = f"{ART}/log_reranker_rulebased"
    os.makedirs(out_dir, exist_ok=True)

    rule_config = {
        "type": "rule_based",
        "signals": [
            "token_overlap",
            "error_keywords",
            "timeout_keywords",
            "auth_keywords",
            "stack_trace_indicators"
        ],
        "reason": f"Insufficient labeled pairs (found {len(log_df)} rows). Using heuristic scoring."
    }

    with open(f"{out_dir}/config.json", "w") as f:
        json.dump(rule_config, f, indent=2)

    summary_metrics["log_rerank_macro_f1"] = None
    summary_metrics["log_rerank_precision_at_20"] = None
    summary_metrics["log_rerank_mode"] = "rule_based_fallback"

    print(f"✓ Rule-based reranker configured. Config saved to {out_dir}/config.json")
    print(f"  This will be used in inference to score log relevance heuristically.")

print("=== LOG RELEVANCE RERANKER COMPLETE ===\n")

## Section 6: Semantic Retrieval — BGE-base-en-v1.5 + FAISS


In [ ]:
# ============================================================
# 5) RETRIEVAL EVAL — BGE + FAISS (Recall@K, MRR@10)
# ============================================================
# Needs: retrieval_query, retrieval_doc, retrieval_group_id, retrieval_relevant (0/1)
ret_df = df[(df["retrieval_query"].str.len()>0) & (df["retrieval_doc"].str.len()>0)].copy()
if len(ret_df) > 500 and ret_df["retrieval_relevant"].nunique() > 1:
    # use test slice only to evaluate retrieval behavior
    _, _, te = time_split(ret_df)

    embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

    docs = te["retrieval_doc"].astype(str).tolist()
    doc_emb = embed_model.encode(docs, normalize_embeddings=True, show_progress_bar=True)
    doc_emb = np.asarray(doc_emb, dtype="float32")

    index = faiss.IndexFlatIP(doc_emb.shape[1])
    index.add(doc_emb)

    # group by query id
    groups = te.groupby("retrieval_group_id", dropna=False)
    recall_at_5_list = []
    mrr10_list = []

    # map row->global doc idx
    te = te.reset_index(drop=True)
    # query once per group using first query text
    for gid, g in groups:
        qtext = str(g["retrieval_query"].iloc[0])
        qemb = embed_model.encode([qtext], normalize_embeddings=True)
        qemb = np.asarray(qemb, dtype="float32")

        D, I = index.search(qemb, 10)
        ranked_idx = I[0].tolist()

        # relevant doc indices in global te frame
        rel_idx = set(g[g["retrieval_relevant"].astype(int)==1].index.tolist())

        top5 = set(ranked_idx[:5])
        hit = 1.0 if len(rel_idx & top5) > 0 else 0.0
        recall_at_5_list.append(hit)

        rr = 0.0
        for rank, di in enumerate(ranked_idx, start=1):
            if di in rel_idx:
                rr = 1.0/rank
                break
        mrr10_list.append(rr)

    r5 = float(np.mean(recall_at_5_list)) if recall_at_5_list else 0.0
    mrr10 = float(np.mean(mrr10_list)) if mrr10_list else 0.0

    summary_metrics["retrieval_recall_at_5"] = r5
    summary_metrics["retrieval_mrr_at_10"] = mrr10

    out_dir = f"{ART}/retrieval_bge"
    os.makedirs(out_dir, exist_ok=True)
    faiss.write_index(index, f"{out_dir}/faiss.index")
    with open(f"{out_dir}/metrics.json", "w") as f:
        json.dump({"recall_at_5": r5, "mrr_at_10": mrr10}, f, indent=2)

    print("Retrieval Recall@5:", round(r5,4), "| MRR@10:", round(mrr10,4))
else:
    print("Skipped retrieval eval (need retrieval_* columns with enough rows).")

## Section 7: Incident Risk Prediction — LightGBM Binary


In [ ]:
# ============================================================
# 6) INCIDENT RISK — LightGBM binary
# ============================================================
risk_df = df[df["incident_in_next_30m"].isin([0,1])].copy()
if len(risk_df) > 400 and risk_df["incident_in_next_30m"].nunique() > 1:
    tr, va, te = time_split(risk_df)

    feat_cols = ["error_count","timeout_count","fatal_count","affected_users","downtime_minutes","auth_error_count"]
    for c in feat_cols:
        if c not in risk_df.columns:
            risk_df[c] = 0

    # include text tfidf + numeric + categorical
    for frame in [tr,va,te]:
        frame["text_merged"] = (frame["summary"].astype(str) + " " + frame["description"].astype(str) +
                                " " + frame["comments_text"].astype(str) + " " + frame["top_error_lines"].astype(str))

    pre = ColumnTransformer([
        ("txt", TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=3), "text_merged"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["env","service"]),
        ("num", "passthrough", feat_cols),
    ])

    clf = lgb.LGBMClassifier(
        objective="binary",
        learning_rate=0.03,
        num_leaves=127,
        min_data_in_leaf=100,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=1,
        reg_alpha=1.0,
        reg_lambda=2.0,
        n_estimators=1000,
        random_state=42
    )

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    ytr = tr["incident_in_next_30m"].astype(int).values
    yte = te["incident_in_next_30m"].astype(int).values

    pipe.fit(tr[["text_merged","env","service"]+feat_cols], ytr)
    prob = pipe.predict_proba(te[["text_merged","env","service"]+feat_cols])[:,1]
    pr_auc = float(average_precision_score(yte, prob))

    # Recall at top alert budget (example top 20/day approximated by top 5% samples here)
    k = max(1, int(0.05 * len(prob)))
    idx = np.argsort(-prob)[:k]
    recall_top = float(yte[idx].sum() / max(1, yte.sum()))

    summary_metrics["incident_pr_auc"] = pr_auc
    summary_metrics["incident_recall_top5pct"] = recall_top

    out_dir = f"{ART}/incident_lgbm"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(pipe, f"{out_dir}/model.joblib")
    with open(f"{out_dir}/metrics.json","w") as f:
        json.dump({"pr_auc": pr_auc, "recall_top5pct": recall_top}, f, indent=2)

    print("Incident PR-AUC:", round(pr_auc,4), "| Recall@Top5%:", round(recall_top,4))
else:
    print("Skipped incident model (need incident_in_next_30m binary labels and enough rows).")

## Section 8: Final Inference + Orchestration Layer

Helper loading, unified inference, and orchestration logic from `Final (2).ipynb` cell 8.


In [ ]:
# ============================================================
# 7) FINAL INFERENCE + RCA PIPELINE
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import torch

# Optional imports used only if corresponding artifacts/models exist
try:
    import faiss
except Exception:
    faiss = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ----------------------------
# Paths
# ----------------------------
BASE = NOTEBOOK_BASE
ART = f"{BASE}/artifacts_all"

# ----------------------------
# Helpers
# ----------------------------
def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x)

def make_ticket_text(ticket):
    return (
        safe_str(ticket.get("summary")) + " [SEP] " +
        safe_str(ticket.get("description")) + " [SEP] " +
        safe_str(ticket.get("comments_text")) + " [SEP] " +
        safe_str(ticket.get("top_error_lines"))
    )

def make_emotion_text(ticket):
    return (
        safe_str(ticket.get("summary")) + " [SEP] " +
        safe_str(ticket.get("description")) + " [SEP] " +
        safe_str(ticket.get("comments_text"))
    )

def make_priority_text(ticket):
    return (
        safe_str(ticket.get("summary")) + " " +
        safe_str(ticket.get("description")) + " " +
        safe_str(ticket.get("comments_text")) + " " +
        safe_str(ticket.get("top_error_lines"))
    )

def softmax_np(x):
    x = np.array(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def infer_transformer_label(text, model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()

    enc = tok(text, truncation=True, max_length=384, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)
        logits = out.logits.cpu().numpy()[0]
    probs = softmax_np(logits)
    pred_idx = int(np.argmax(probs))
    return pred_idx, probs

# ----------------------------
# Load artifacts if present
# ----------------------------
artifacts = {}

# Category
if os.path.exists(f"{ART}/classification_deberta"):
    artifacts["category_mode"] = "deberta"
    artifacts["category_model_dir"] = f"{ART}/classification_deberta"
    artifacts["category_label_encoder"] = joblib.load(f"{ART}/classification_deberta/label_encoder.joblib")
elif os.path.exists(f"{ART}/classification_tfidf_lr/model.joblib"):
    artifacts["category_mode"] = "tfidf_lr"
    artifacts["category_model"] = joblib.load(f"{ART}/classification_tfidf_lr/model.joblib")
    artifacts["category_label_encoder"] = joblib.load(f"{ART}/classification_tfidf_lr/label_encoder.joblib")

# Priority
if os.path.exists(f"{ART}/priority_lgbm/model.joblib"):
    artifacts["priority_model"] = joblib.load(f"{ART}/priority_lgbm/model.joblib")
    artifacts["priority_label_encoder"] = joblib.load(f"{ART}/priority_lgbm/label_encoder.joblib")

# Emotion
if os.path.exists(f"{ART}/emotion_distilroberta"):
    emo_le_path = f"{ART}/emotion_distilroberta/label_encoder.joblib"
    if os.path.exists(emo_le_path):
        artifacts["emotion_model_dir"] = f"{ART}/emotion_distilroberta"
        artifacts["emotion_label_encoder"] = joblib.load(emo_le_path)

# Incident
if os.path.exists(f"{ART}/incident_lgbm/model.joblib"):
    artifacts["incident_model"] = joblib.load(f"{ART}/incident_lgbm/model.joblib")

# Retrieval
if faiss is not None and os.path.exists(f"{ART}/retrieval_bge/faiss.index"):
    artifacts["retrieval_index"] = faiss.read_index(f"{ART}/retrieval_bge/faiss.index")

print("Loaded artifacts:", list(artifacts.keys()))

# ----------------------------
# Prediction functions
# ----------------------------
def predict_category(ticket):
    if "category_mode" not in artifacts:
        return {"label": None, "confidence": None, "source": "missing_model"}

    text = make_ticket_text(ticket)

    if artifacts["category_mode"] == "deberta":
        pred_idx, probs = infer_transformer_label(text, artifacts["category_model_dir"])
        label = artifacts["category_label_encoder"].inverse_transform([pred_idx])[0]
        return {
            "label": label,
            "confidence": float(np.max(probs)),
            "source": "deberta"
        }

    if artifacts["category_mode"] == "tfidf_lr":
        clf = artifacts["category_model"]
        le = artifacts["category_label_encoder"]
        pred_idx = clf.predict([text])[0]
        label = le.inverse_transform([pred_idx])[0]

        confidence = None
        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba([text])[0]
            confidence = float(np.max(probs))

        return {
            "label": label,
            "confidence": confidence,
            "source": "tfidf_lr"
        }

def predict_priority(ticket):
    if "priority_model" not in artifacts:
        return {"label": None, "confidence": None, "source": "missing_model"}

    num_cols = [
        "affected_users", "downtime_minutes", "error_count",
        "fatal_count", "timeout_count", "auth_error_count"
    ]
    cat_cols = ["env", "service"]

    row = {
        "text_merged": make_priority_text(ticket),
        "env": safe_str(ticket.get("env", "unknown")),
        "service": safe_str(ticket.get("service", "unknown")),
    }
    for c in num_cols:
        row[c] = float(ticket.get(c, 0) or 0)

    X = pd.DataFrame([row])
    pipe = artifacts["priority_model"]
    le = artifacts["priority_label_encoder"]

    pred_idx = pipe.predict(X)[0]
    label = le.inverse_transform([pred_idx])[0]

    confidence = None
    if hasattr(pipe, "predict_proba"):
        probs = pipe.predict_proba(X)[0]
        confidence = float(np.max(probs))

    return {
        "label": label,
        "confidence": confidence,
        "source": "lightgbm"
    }

def predict_emotion(ticket):
    if "emotion_model_dir" not in artifacts:
        return {"label": None, "confidence": None, "source": "missing_model"}

    text = make_emotion_text(ticket)
    pred_idx, probs = infer_transformer_label(text, artifacts["emotion_model_dir"])
    label = artifacts["emotion_label_encoder"].inverse_transform([pred_idx])[0]

    return {
        "label": label,
        "confidence": float(np.max(probs)),
        "source": "distilroberta"
    }

def predict_incident_risk(ticket):
    if "incident_model" not in artifacts:
        return {"score": None, "source": "missing_model"}

    feat_cols = [
        "error_count", "timeout_count", "fatal_count",
        "affected_users", "downtime_minutes", "auth_error_count"
    ]

    row = {
        "text_merged": make_priority_text(ticket),
        "env": safe_str(ticket.get("env", "unknown")),
        "service": safe_str(ticket.get("service", "unknown")),
    }
    for c in feat_cols:
        row[c] = float(ticket.get(c, 0) or 0)

    X = pd.DataFrame([row])
    pipe = artifacts["incident_model"]
    score = float(pipe.predict_proba(X)[0][1])

    return {
        "score": score,
        "source": "lightgbm"
    }

# ----------------------------
# Optional log reranking (rule-based placeholder if no model)
# ----------------------------
def rank_logs(ticket, candidate_logs, top_k=5):
    text = make_ticket_text(ticket).lower()

    scored = []
    for log in candidate_logs:
        log_s = safe_str(log)
        score = 0

        # simple lexical overlap heuristic
        for tok in set(text.split()):
            if len(tok) > 3 and tok in log_s.lower():
                score += 1

        if "error" in log_s.lower():
            score += 1
        if "exception" in log_s.lower():
            score += 1
        if "timeout" in log_s.lower():
            score += 1
        if "failed" in log_s.lower():
            score += 1

        scored.append((log_s, score))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    return [{"log_line": s[0], "score": float(s[1])} for s in scored[:top_k]]

# ----------------------------
# Optional retrieval placeholder
# ----------------------------
def retrieve_docs(ticket, doc_df=None, top_k=5):
    if doc_df is None or len(doc_df) == 0:
        return []

    query = make_ticket_text(ticket).lower()
    rows = []

    for _, r in doc_df.iterrows():
        doc = safe_str(r.get("retrieval_doc", ""))
        score = 0
        for tok in set(query.split()):
            if len(tok) > 3 and tok in doc.lower():
                score += 1
        rows.append((doc, score))

    rows = sorted(rows, key=lambda x: x[1], reverse=True)
    return [{"doc": x[0], "score": float(x[1])} for x in rows[:top_k]]


# ----------------------------
# Unified inference
# ----------------------------
def predict_ticket(ticket, candidate_logs=None, retrieval_df=None):
    if candidate_logs is None:
        candidate_logs = []

    category = predict_category(ticket)
    priority = predict_priority(ticket)
    emotion = predict_emotion(ticket)
    incident_risk = predict_incident_risk(ticket)
    top_logs = rank_logs(ticket, candidate_logs, top_k=5)
    retrieved_docs = retrieve_docs(ticket, retrieval_df, top_k=5)

    bundle = {
        "category": category,
        "priority": priority,
        "emotion": emotion,
        "incident_risk": incident_risk,
        "top_logs": top_logs,
        "retrieved_docs": retrieved_docs
    }

    bundle["rca"] = generate_rca(ticket, bundle)
    return bundle


## Section 9: Rule-Based RCA Generator

Explainable RCA logic from the final inference notebook plus the operational RCA task.


In [ ]:
# ----------------------------
# RCA generation
# ----------------------------
def generate_rca(ticket, result_bundle):
    findings = []

    category = result_bundle.get("category", {}).get("label")
    priority = result_bundle.get("priority", {}).get("label")
    emotion = result_bundle.get("emotion", {}).get("label")
    risk = result_bundle.get("incident_risk", {}).get("score")

    err = float(ticket.get("error_count", 0) or 0)
    fatal = float(ticket.get("fatal_count", 0) or 0)
    timeout = float(ticket.get("timeout_count", 0) or 0)
    auth = float(ticket.get("auth_error_count", 0) or 0)
    affected = float(ticket.get("affected_users", 0) or 0)
    downtime = float(ticket.get("downtime_minutes", 0) or 0)
    env = safe_str(ticket.get("env", "unknown"))
    service = safe_str(ticket.get("service", "unknown"))

    likely_causes = []

    if auth > 0:
        likely_causes.append("authentication or authorization failure")
    if timeout > 0:
        likely_causes.append("timeout or downstream service latency")
    if fatal > 0:
        likely_causes.append("fatal application/runtime failure")
    if err > 50:
        likely_causes.append("high error-volume service degradation")
    if env.lower() in {"prod", "production"} and downtime > 0:
        likely_causes.append("production service disruption")

    if not likely_causes and category:
        likely_causes.append(f"issue related to category '{category}'")

    findings.append(f"Service: {service or 'unknown'} in env: {env or 'unknown'}.")
    if priority:
        findings.append(f"Predicted priority is {priority}.")
    if emotion:
        findings.append(f"Reporter/user emotion appears {emotion}.")
    if risk is not None:
        findings.append(f"Predicted near-term incident risk score is {risk:.3f}.")
    if affected > 0:
        findings.append(f"Estimated affected users: {int(affected)}.")
    if downtime > 0:
        findings.append(f"Observed downtime minutes: {int(downtime)}.")

    top_logs = result_bundle.get("top_logs", [])
    if top_logs:
        findings.append("Most relevant log evidence:")
        for item in top_logs[:3]:
            findings.append(f"- {item['log_line']}")

    retrieved = result_bundle.get("retrieved_docs", [])
    if retrieved:
        findings.append("Most relevant retrieved references:")
        for item in retrieved[:2]:
            findings.append(f"- {item['doc'][:200]}")

    summary = {
        "likely_root_causes": likely_causes,
        "evidence_summary": findings,
        "recommended_next_steps": [
            "Inspect the top reranked log lines for the first failing component.",
            "Check recent deployments/config changes for the predicted service/category.",
            "Validate dependency health and timeout/authentication paths.",
            "Compare with retrieved similar incidents or support documents."
        ]
    }
    return summary


## 8. RCA Task (`tasks/rca.py`)

Generates a Root Cause Analysis via LLM and posts it to Jira.

In [ ]:
"""Generate Root Cause Analysis via LLM and post to Jira."""


# ---------------------------------------------------------------------------
# LLM client helper
# ---------------------------------------------------------------------------

def _call_llm(prompt: str) -> str:
    """Call OpenAI or Azure OpenAI and return the response text."""
    # Prefer Azure OpenAI if endpoint is configured, otherwise fall back to OpenAI
    if AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY:
        return _call_azure_openai(prompt)
    if OPENAI_API_KEY:
        return _call_openai(prompt)
    raise RuntimeError("No LLM credentials configured (OPENAI_API_KEY or AZURE_OPENAI_*).")


def _call_openai(prompt: str) -> str:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=2048,
    )
    return resp.choices[0].message.content.strip()


def _call_azure_openai(prompt: str) -> str:
    from openai import AzureOpenAI

    client = AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
    )
    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=2048,
    )
    return resp.choices[0].message.content.strip()


# ---------------------------------------------------------------------------
# RCA prompt builder
# ---------------------------------------------------------------------------

def _build_rca_prompt(
    ticket: dict,
    similar_tickets: list[dict],
    confluence_pages: list[dict],
    top_log_lines: list[str],
    incident_risk: float,
) -> str:
    similar_text = "\n".join(
        f"- [{t.get('metadata', {}).get('key', 'N/A')}] "
        f"(similarity {t.get('similarity', 0):.0%}): {t.get('text', '')[:300]}"
        for t in similar_tickets
    ) or "None found."

    confluence_text = "\n".join(
        f"- [{p.get('title', '')}]({p.get('url', '')}): {p.get('excerpt', '')[:300]}"
        for p in confluence_pages
    ) or "None found."

    logs_text = "\n".join(top_log_lines[:20]) or "No logs retrieved."

    return f"""You are an expert Site Reliability Engineer performing a Root Cause Analysis.

## Ticket Details
**Key**: {ticket.get('key', 'N/A')}
**Summary**: {ticket.get('summary', '')}
**Description**: {ticket.get('description', '')}
**Category**: {ticket.get('category', '')}
**Priority**: {ticket.get('priority', '')}
**Service**: {ticket.get('service', '')}
**Environment**: {ticket.get('env', '')}
**Incident Risk Score**: {incident_risk:.0%}

## Similar Past Tickets
{similar_text}

## Relevant Confluence Knowledge Base
{confluence_text}

## Top Relevant Log Lines
{logs_text}

## Instructions
Produce a structured RCA with the following sections:

1. **Root Cause** — What is the primary technical cause?
2. **Contributing Factors** — Secondary conditions that amplified the issue.
3. **Impact** — Services/users affected and severity.
4. **Timeline** — Inferred sequence of events leading to the issue.
5. **Resolution Steps** — Concrete actions to resolve the issue now.
6. **Prevention / Follow-up** — Long-term fixes, monitoring improvements, runbook updates.

Be concise, technical, and actionable. Base your analysis strictly on the evidence provided.
"""


# ---------------------------------------------------------------------------
# Jira comment helper
# ---------------------------------------------------------------------------

def _post_rca_jira_comment(ticket_key: str, body: str) -> None:
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        logger.warning("Jira credentials not configured — skipping comment post")
        return

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/comment"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "body": {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [{"type": "text", "text": body}],
                }
            ],
        }
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to post Jira comment: %s %s", resp.status_code, resp.text)


# ---------------------------------------------------------------------------
# RCA generation
# ---------------------------------------------------------------------------

def generate_rca(ticket: dict, context: dict) -> dict:
    """Generate an RCA and post it to Jira.

    Args:
        ticket: classified ticket dict (key, summary, description, category, priority, …).
        context: combined output of classify, log_fetch, and search tasks:
            {similar_tickets, confluence_pages, top_log_lines, incident_risk}.

    Returns:
        dict with ``rca_text`` and ``confluence_created`` flag.
    """
    ticket_key = ticket.get("key", "UNKNOWN")
    logger.info("Generating RCA for %s", ticket_key)

    similar = context.get("similar_tickets", [])
    confluence = context.get("confluence_pages", [])
    logs = context.get("top_log_lines", [])
    incident_risk = float(context.get("incident_risk", 0.0))

    prompt = _build_rca_prompt(ticket, similar, confluence, logs, incident_risk)
    rca_text = _call_llm(prompt)

    # Post RCA to Jira
    header = (
        f"\U0001f50d *Root Cause Analysis \u2014 {ticket_key}*\n"
        f"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
    )
    _post_rca_jira_comment(ticket_key, header + rca_text)

    # If no Confluence pages found, create a knowledge article
    confluence_created = False
    if not confluence:
        logger.info("No Confluence pages found — creating knowledge article for %s", ticket_key)
        create_kb_article(ticket=ticket, rca_text=rca_text)
        confluence_created = True

    return {"rca_text": rca_text, "confluence_created": confluence_created}


print("RCA functions defined.")

## Section 10: AIOps Pipeline — Configuration, Celery, Model Loader, Tasks, FastAPI

Operational pipeline cells from `aiops_notebook.ipynb` (excluding the RCA task already placed above).


## 2. Configuration (`config.py`)

Central configuration loaded from environment variables.

In [ ]:
"""Central configuration loaded from environment variables."""

from __future__ import annotations

import os
from dotenv import load_dotenv

load_dotenv()


def _require(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise RuntimeError(f"Required environment variable '{key}' is not set.")
    return val


# ---------------------------------------------------------------------------
# Jira
# ---------------------------------------------------------------------------
JIRA_BASE_URL: str = os.getenv("JIRA_BASE_URL", "")          # e.g. https://myorg.atlassian.net
JIRA_USER: str = os.getenv("JIRA_USER", "")                   # service-account email
JIRA_API_TOKEN: str = os.getenv("JIRA_API_TOKEN", "")
JIRA_PROJECT_KEY: str = os.getenv("JIRA_PROJECT_KEY", "OPS")

# ---------------------------------------------------------------------------
# Confluence
# ---------------------------------------------------------------------------
CONFLUENCE_BASE_URL: str = os.getenv("CONFLUENCE_BASE_URL", JIRA_BASE_URL)
CONFLUENCE_SPACE_KEY: str = os.getenv("CONFLUENCE_SPACE_KEY", "KB")
CONFLUENCE_PARENT_PAGE_ID: str = os.getenv("CONFLUENCE_PARENT_PAGE_ID", "")

# ---------------------------------------------------------------------------
# Azure Storage (mobile logs)
# ---------------------------------------------------------------------------
AZURE_STORAGE_CONNECTION_STRING: str = os.getenv("AZURE_STORAGE_CONNECTION_STRING", "")
AZURE_FILE_SHARE_NAME: str = os.getenv("AZURE_FILE_SHARE_NAME", "mobile-logs")
AZURE_LOG_DIRECTORY: str = os.getenv("AZURE_LOG_DIRECTORY", "")

# ---------------------------------------------------------------------------
# Datadog (server logs + monitoring)
# ---------------------------------------------------------------------------
DATADOG_API_KEY: str = os.getenv("DATADOG_API_KEY", "")
DATADOG_APP_KEY: str = os.getenv("DATADOG_APP_KEY", "")
DATADOG_SITE: str = os.getenv("DATADOG_SITE", "datadoghq.com")
DATADOG_LOG_LOOKBACK_HOURS: int = int(os.getenv("DATADOG_LOG_LOOKBACK_HOURS", "6"))

# ---------------------------------------------------------------------------
# OpenAI / Azure OpenAI
# ---------------------------------------------------------------------------
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL: str = os.getenv("OPENAI_MODEL", "gpt-4o")
AZURE_OPENAI_ENDPOINT: str = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_API_KEY: str = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT: str = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
AZURE_OPENAI_API_VERSION: str = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01")

# ---------------------------------------------------------------------------
# Email / SMTP
# ---------------------------------------------------------------------------
SMTP_HOST: str = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT: int = int(os.getenv("SMTP_PORT", "587"))
SMTP_USER: str = os.getenv("SMTP_USER", "")
SMTP_PASSWORD: str = os.getenv("SMTP_PASSWORD", "")
ALERT_EMAIL_RECIPIENTS: list[str] = [
    e.strip()
    for e in os.getenv("ALERT_EMAIL_RECIPIENTS", "").split(",")
    if e.strip()
]

# ---------------------------------------------------------------------------
# Celery / Redis
# ---------------------------------------------------------------------------
REDIS_URL: str = os.getenv("REDIS_URL", "redis://localhost:6379/0")

# ---------------------------------------------------------------------------
# Model artifacts
# ---------------------------------------------------------------------------
ARTIFACTS_DIR: str = os.getenv("ARTIFACTS_DIR", "artifacts")

# ---------------------------------------------------------------------------
# Log alert thresholds
# ---------------------------------------------------------------------------
ERROR_COUNT_THRESHOLD: int = int(os.getenv("ERROR_COUNT_THRESHOLD", "50"))
WARNING_COUNT_THRESHOLD: int = int(os.getenv("WARNING_COUNT_THRESHOLD", "200"))
FATAL_COUNT_THRESHOLD: int = int(os.getenv("FATAL_COUNT_THRESHOLD", "5"))

# ---------------------------------------------------------------------------
# Retrieval
# ---------------------------------------------------------------------------
TOP_K_SIMILAR_TICKETS: int = int(os.getenv("TOP_K_SIMILAR_TICKETS", "5"))
TOP_K_LOG_LINES: int = int(os.getenv("TOP_K_LOG_LINES", "20"))
TOP_K_CONFLUENCE: int = int(os.getenv("TOP_K_CONFLUENCE", "3"))

# ChromaDB
CHROMA_PERSIST_DIR: str = os.getenv("CHROMA_PERSIST_DIR", "/tmp/aiops_chroma")

print("Configuration loaded.")

## 3. Celery Application (`celery_app.py`)

Celery application instance and task routing.

In [ ]:
"""Celery application instance and task routing."""

from celery import Celery

celery_app = Celery(
    "aiops",
    broker=REDIS_URL,
    backend=REDIS_URL,
    include=[
        "aiops.tasks.classify",
        "aiops.tasks.log_fetch",
        "aiops.tasks.search",
        "aiops.tasks.rca",
        "aiops.tasks.notify",
        "aiops.tasks.confluence_kb",
    ],
)

celery_app.conf.update(
    task_serializer="json",
    result_serializer="json",
    accept_content=["json"],
    timezone="UTC",
    enable_utc=True,
    task_acks_late=True,
    worker_prefetch_multiplier=1,
    task_routes={
        "aiops.tasks.classify.*": {"queue": "classify"},
        "aiops.tasks.log_fetch.*": {"queue": "logs"},
        "aiops.tasks.search.*": {"queue": "search"},
        "aiops.tasks.rca.*": {"queue": "rca"},
        "aiops.tasks.notify.*": {"queue": "notify"},
        "aiops.tasks.confluence_kb.*": {"queue": "confluence"},
    },
    task_default_retry_delay=30,   # seconds
    task_max_retries=3,
)

print("Celery app configured.")

## 4. Model Loader (`models/loader.py`)

Loads trained model artifacts once at process startup.

Artifacts expected under `ARTIFACTS_DIR`:
- `classification_deberta/` — DeBERTa seq-classification model + tokenizer + `label_encoder.joblib`
- `priority_lgbm/` — LightGBM booster (`priority_model.txt`) + `tfidf_vectorizer.joblib` + `priority_label_encoder.joblib` + `ohe_encoder.joblib`
- `log_reranker/` — cross-encoder checkpoint
- `incident_risk/` — LightGBM booster (`incident_risk_model.txt`)

In [ ]:
"""Load trained model artifacts once at process startup."""

import logging
from pathlib import Path
from typing import Any

import joblib

logger = logging.getLogger(__name__)

_REGISTRY: dict[str, Any] = {}


def _art(name: str) -> Path:
    return Path(ARTIFACTS_DIR) / name


def _load_category_model() -> None:
    """Load DeBERTa category classifier (optional — falls back to None)."""
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch

        path = _art("classification_deberta")
        if not path.exists():
            logger.warning("Category model not found at %s — skipping", path)
            return

        tok = AutoTokenizer.from_pretrained(str(path))
        model = AutoModelForSequenceClassification.from_pretrained(str(path))
        model.eval()
        le = joblib.load(path / "label_encoder.joblib")

        _REGISTRY["cat_tokenizer"] = tok
        _REGISTRY["cat_model"] = model
        _REGISTRY["cat_le"] = le
        logger.info("Category model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load category model: %s", exc)


def _load_priority_model() -> None:
    """Load LightGBM priority classifier."""
    try:
        import lightgbm as lgb

        path = _art("priority_lgbm")
        if not path.exists():
            logger.warning("Priority model not found at %s — skipping", path)
            return

        booster = lgb.Booster(model_file=str(path / "priority_model.txt"))
        tfidf = joblib.load(path / "tfidf_vectorizer.joblib")
        le = joblib.load(path / "priority_label_encoder.joblib")
        ohe = joblib.load(path / "ohe_encoder.joblib") if (path / "ohe_encoder.joblib").exists() else None

        _REGISTRY["prio_booster"] = booster
        _REGISTRY["prio_tfidf"] = tfidf
        _REGISTRY["prio_le"] = le
        _REGISTRY["prio_ohe"] = ohe
        logger.info("Priority model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load priority model: %s", exc)


def _load_log_reranker() -> None:
    """Load cross-encoder log re-ranker."""
    try:
        from sentence_transformers.cross_encoder import CrossEncoder

        path = _art("log_reranker")
        if not path.exists():
            logger.warning("Log reranker not found at %s — skipping", path)
            return

        _REGISTRY["log_reranker"] = CrossEncoder(str(path))
        logger.info("Log reranker loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load log reranker: %s", exc)


def _load_incident_risk() -> None:
    """Load LightGBM incident-risk predictor."""
    try:
        import lightgbm as lgb

        path = _art("incident_risk")
        if not path.exists():
            logger.warning("Incident risk model not found at %s — skipping", path)
            return

        _REGISTRY["incident_booster"] = lgb.Booster(
            model_file=str(path / "incident_risk_model.txt")
        )
        logger.info("Incident risk model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load incident risk model: %s", exc)


def load_all() -> None:
    """Call once at application startup to populate the model registry."""
    if _REGISTRY:
        return  # already loaded
    _load_category_model()
    _load_priority_model()
    _load_log_reranker()
    _load_incident_risk()
    logger.info("Model registry ready: %s", list(_REGISTRY.keys()))


def get_model(name: str) -> Any:
    """Retrieve a loaded artefact by name; returns None if unavailable."""
    return _REGISTRY.get(name)


# Load all models
load_all()
print("Model registry:", list(_REGISTRY.keys()) or "(no artifacts found)")

## 5. Classify Task (`tasks/classify.py`)

Classifies a Jira ticket (priority + category) and posts results back to Jira.

In [ ]:
"""Classify a Jira ticket (priority + category) and post back."""

import numpy as np


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _build_text(ticket: dict) -> str:
    parts = [
        ticket.get("summary", ""),
        ticket.get("description", ""),
        ticket.get("comments_text", ""),
        ticket.get("top_error_lines", ""),
    ]
    return " [SEP] ".join(p for p in parts if p)


def _classify_category(text: str) -> str | None:
    """Run DeBERTa category model; returns label string or None."""
    import torch

    model = get_model("cat_model")
    tok = get_model("cat_tokenizer")
    le = get_model("cat_le")
    if model is None or tok is None or le is None:
        return None

    try:
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=384)
        with torch.no_grad():
            logits = model(**inputs).logits
        idx = int(torch.argmax(logits, dim=-1).item())
        return str(le.inverse_transform([idx])[0])
    except Exception as exc:
        logger.warning("Category inference failed: %s", exc)
        return None


def _classify_priority(ticket: dict) -> str | None:
    """Run LightGBM priority model; returns P1–P5 string or None."""
    booster = get_model("prio_booster")
    tfidf = get_model("prio_tfidf")
    le = get_model("prio_le")
    if booster is None or tfidf is None or le is None:
        return None

    try:
        text = _build_text(ticket)
        tfidf_feat = tfidf.transform([text]).toarray()

        numeric_cols = [
            "affected_users", "downtime_minutes", "error_count",
            "fatal_count", "timeout_count", "auth_error_count",
        ]
        num_feat = np.array([[float(ticket.get(c, 0) or 0) for c in numeric_cols]])

        ohe = get_model("prio_ohe")
        cat_cols = ["env", "service"]
        cat_vals = [[str(ticket.get(c, "unknown")) for c in cat_cols]]
        if ohe is not None:
            cat_feat = ohe.transform(cat_vals)
        else:
            cat_feat = np.zeros((1, 1))

        X = np.hstack([tfidf_feat, num_feat, cat_feat])
        proba = booster.predict(X)
        idx = int(np.argmax(proba, axis=1)[0])
        return str(le.inverse_transform([idx])[0])
    except Exception as exc:
        logger.warning("Priority inference failed: %s", exc)
        return None


def _predict_incident_risk(ticket: dict) -> float:
    """Return probability [0,1] that this ticket precedes an incident."""
    booster = get_model("incident_booster")
    if booster is None:
        return 0.0
    try:
        tfidf = get_model("prio_tfidf")
        text = _build_text(ticket)
        tfidf_feat = tfidf.transform([text]).toarray() if tfidf else np.zeros((1, 100))
        numeric_cols = [
            "affected_users", "downtime_minutes", "error_count",
            "fatal_count", "timeout_count", "auth_error_count",
        ]
        num_feat = np.array([[float(ticket.get(c, 0) or 0) for c in numeric_cols]])
        X = np.hstack([tfidf_feat, num_feat])
        proba = booster.predict(X)
        return float(proba[0]) if proba.ndim == 1 else float(proba[0, 1])
    except Exception as exc:
        logger.warning("Incident risk inference failed: %s", exc)
        return 0.0


def _post_jira_comment(ticket_key: str, body: str) -> None:
    """Post a plain-text comment to a Jira issue."""
    import requests
    from requests.auth import HTTPBasicAuth

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/comment"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "body": {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [{"type": "text", "text": body}],
                }
            ],
        }
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to post Jira comment: %s %s", resp.status_code, resp.text)


def _update_jira_fields(ticket_key: str, fields: dict) -> None:
    """Update arbitrary Jira issue fields."""
    import requests
    from requests.auth import HTTPBasicAuth

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    resp = requests.put(url, json={"fields": fields}, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to update Jira fields: %s %s", resp.status_code, resp.text)


def classify_ticket(ticket: dict) -> dict:
    """Classify a ticket and post results back to Jira.

    Args:
        ticket: dict containing at minimum ``key``, ``summary``, ``description``.

    Returns:
        dict with ``category``, ``priority``, ``incident_risk``.
    """
    load_all()

    ticket_key = ticket.get("key", "UNKNOWN")
    logger.info("Classifying ticket %s", ticket_key)

    # Acknowledge receipt
    _post_jira_comment(
        ticket_key,
        "\U0001f916 AI Ops: Ticket received. Classification and RCA analysis in progress\u2026",
    )

    text = _build_text(ticket)
    category = _classify_category(text) or ticket.get("final_category", "Unknown")
    priority = _classify_priority(ticket) or ticket.get("final_priority", "P3")
    incident_risk = _predict_incident_risk(ticket)

    # Post classification summary
    comment = (
        f"\U0001f4ca *Classification Results*\n"
        f"\u2022 Category: {category}\n"
        f"\u2022 Priority: {priority}\n"
        f"\u2022 Incident Risk Score: {incident_risk:.0%}\n"
    )
    _post_jira_comment(ticket_key, comment)

    # Update Jira priority field if we have a valid mapping
    priority_map = {"P1": "Highest", "P2": "High", "P3": "Medium", "P4": "Low", "P5": "Lowest"}
    jira_priority = priority_map.get(priority, "Medium")
    _update_jira_fields(ticket_key, {"priority": {"name": jira_priority}})

    result = {"category": category, "priority": priority, "incident_risk": incident_risk}
    logger.info("Ticket %s classified: %s", ticket_key, result)
    return result


print("Classify functions defined.")

## 6. Log Fetch Task (`tasks/log_fetch.py`)

Fetches logs from Azure File Share (mobile) or Datadog (server) and re-ranks them.

In [ ]:
"""Fetch logs from Azure File Share (mobile) or Datadog (server)."""

import time
from datetime import datetime, timezone, timedelta


# ---------------------------------------------------------------------------
# Log re-ranking helper
# ---------------------------------------------------------------------------

def _rerank_logs(query: str, log_lines: list[str]) -> list[str]:
    """Score log lines against the ticket query and return top-k."""
    reranker = get_model("log_reranker")
    if reranker is None or not log_lines:
        return log_lines[:TOP_K_LOG_LINES]

    try:
        pairs = [(query, line) for line in log_lines]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, log_lines), key=lambda x: x[0], reverse=True)
        return [line for _, line in ranked[:TOP_K_LOG_LINES]]
    except Exception as exc:
        logger.warning("Log reranking failed: %s", exc)
        return log_lines[:TOP_K_LOG_LINES]


# ---------------------------------------------------------------------------
# Azure File Share — mobile logs
# ---------------------------------------------------------------------------

def _fetch_azure_logs(service: str, env: str, lookback_hours: int = 6) -> list[str]:
    """Fetch log lines from Azure File Share for mobile issues."""
    try:
        from azure.storage.fileshare import ShareServiceClient

        conn_str = AZURE_STORAGE_CONNECTION_STRING
        if not conn_str:
            logger.warning("AZURE_STORAGE_CONNECTION_STRING not set — skipping mobile logs")
            return []

        svc = ShareServiceClient.from_connection_string(conn_str)
        share_client = svc.get_share_client(AZURE_FILE_SHARE_NAME)

        directory = AZURE_LOG_DIRECTORY or service
        dir_client = share_client.get_directory_client(directory)

        cutoff = datetime.now(timezone.utc) - timedelta(hours=lookback_hours)
        lines: list[str] = []

        for item in dir_client.list_directories_and_files():
            if item["is_directory"]:
                continue
            if item.get("last_modified") and item["last_modified"] < cutoff:
                continue

            file_client = dir_client.get_file_client(item["name"])
            content = file_client.download_file().readall().decode("utf-8", errors="replace")
            lines.extend(content.splitlines())

        logger.info("Fetched %d log lines from Azure File Share (service=%s)", len(lines), service)
        return lines
    except Exception as exc:
        logger.error("Azure log fetch failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Datadog — server logs
# ---------------------------------------------------------------------------

def _fetch_datadog_logs(service: str, env: str, lookback_hours: int | None = None) -> list[str]:
    """Fetch log lines from Datadog Logs API for server issues."""
    try:
        import requests

        if not DATADOG_API_KEY:
            logger.warning("DATADOG_API_KEY not set — skipping Datadog logs")
            return []

        hours = lookback_hours or DATADOG_LOG_LOOKBACK_HOURS
        now = datetime.now(timezone.utc)
        start = now - timedelta(hours=hours)

        url = f"https://api.{DATADOG_SITE}/api/v2/logs/events/search"
        headers = {
            "DD-API-KEY": DATADOG_API_KEY,
            "DD-APPLICATION-KEY": DATADOG_APP_KEY,
            "Content-Type": "application/json",
        }
        query_filter = f"service:{service}"
        if env:
            query_filter += f" env:{env}"
        query_filter += " status:(error OR warn OR critical)"

        payload = {
            "filter": {
                "query": query_filter,
                "from": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "to": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
            },
            "sort": "timestamp",
            "page": {"limit": 1000},
        }

        resp = requests.post(url, json=payload, headers=headers, timeout=20)
        resp.raise_for_status()

        data = resp.json()
        events = data.get("data", [])
        lines: list[str] = []
        for evt in events:
            msg = evt.get("attributes", {}).get("message", "")
            if msg:
                lines.append(msg)

        logger.info("Fetched %d log lines from Datadog (service=%s)", len(lines), service)
        return lines
    except Exception as exc:
        logger.error("Datadog log fetch failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Main fetch function
# ---------------------------------------------------------------------------

def fetch_logs(ticket: dict) -> dict:
    """Fetch and re-rank logs relevant to the ticket.

    Determines source (Azure = mobile, Datadog = server) from ticket category.

    Args:
        ticket: dict with ``key``, ``category``, ``service``, ``env``.

    Returns:
        dict with ``top_log_lines`` (list[str]) and ``source`` ("azure" | "datadog" | "none").
    """
    load_all()

    category = (ticket.get("category") or ticket.get("final_category") or "").lower()
    service = ticket.get("service", "")
    env = ticket.get("env", "")
    query = f"{ticket.get('summary', '')} {ticket.get('description', '')}"

    is_mobile = any(kw in category for kw in ("mobile", "android", "ios", "app"))

    if is_mobile:
        raw_lines = _fetch_azure_logs(service, env)
        source = "azure"
    else:
        raw_lines = _fetch_datadog_logs(service, env)
        source = "datadog"

    if not raw_lines:
        return {"top_log_lines": [], "source": "none"}

    top_lines = _rerank_logs(query, raw_lines)

    # Threshold monitoring: count errors/warnings
    error_count = sum(1 for l in raw_lines if "error" in l.lower() or "exception" in l.lower())
    warn_count = sum(1 for l in raw_lines if "warn" in l.lower())
    fatal_count = sum(1 for l in raw_lines if "fatal" in l.lower() or "critical" in l.lower())

    thresholds_exceeded = (
        error_count >= ERROR_COUNT_THRESHOLD
        or warn_count >= WARNING_COUNT_THRESHOLD
        or fatal_count >= FATAL_COUNT_THRESHOLD
    )

    if thresholds_exceeded:
        logger.warning(
            "Log thresholds exceeded for %s — errors=%d, warnings=%d, fatals=%d",
            ticket.get("key"), error_count, warn_count, fatal_count,
        )

    return {
        "top_log_lines": top_lines,
        "source": source,
        "error_count": error_count,
        "warn_count": warn_count,
        "fatal_count": fatal_count,
    }


print("Log fetch functions defined.")

## 7. Search Task (`tasks/search.py`)

Finds similar historical tickets (via ChromaDB) and relevant Confluence pages.

In [ ]:
"""Find similar historical tickets and relevant Confluence pages."""

import re


# ---------------------------------------------------------------------------
# Similar ticket retrieval via ChromaDB
# ---------------------------------------------------------------------------

def _get_chroma_collection():
    """Return (or lazily create) a persistent ChromaDB collection of historical tickets."""
    try:
        import chromadb

        persist_path = CHROMA_PERSIST_DIR
        client = chromadb.PersistentClient(path=persist_path)
        return client.get_or_create_collection("tickets")
    except Exception as exc:
        logger.warning("ChromaDB unavailable: %s", exc)
        return None


def _search_similar_tickets(query: str, top_k: int) -> list[dict]:
    """Query ChromaDB for the top-k most similar historical tickets."""
    collection = _get_chroma_collection()
    if collection is None:
        return []

    try:
        results = collection.query(
            query_texts=[query],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
        tickets = []
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        dists = results.get("distances", [[]])[0]
        for doc, meta, dist in zip(docs, metas, dists):
            tickets.append(
                {
                    "text": doc,
                    "metadata": meta,
                    "similarity": round(1 - dist, 4),
                }
            )
        return tickets
    except Exception as exc:
        logger.error("Similar ticket search failed: %s", exc)
        return []


def index_ticket(ticket: dict) -> None:
    """Upsert a ticket into the ChromaDB collection (call after resolution)."""
    collection = _get_chroma_collection()
    if collection is None:
        return
    try:
        text = (
            f"{ticket.get('summary', '')} "
            f"{ticket.get('description', '')} "
            f"{ticket.get('comments_text', '')}"
        )
        collection.upsert(
            ids=[ticket.get("key", "")],
            documents=[text],
            metadatas=[
                {
                    "key": ticket.get("key", ""),
                    "category": ticket.get("category", ""),
                    "priority": ticket.get("priority", ""),
                    "service": ticket.get("service", ""),
                }
            ],
        )
    except Exception as exc:
        logger.error("Failed to index ticket: %s", exc)


# ---------------------------------------------------------------------------
# Confluence search
# ---------------------------------------------------------------------------

def _search_confluence(query: str, top_k: int) -> list[dict]:
    """Search Confluence using CQL and return top-k matching pages."""
    try:
        import requests
        from requests.auth import HTTPBasicAuth

        if not CONFLUENCE_BASE_URL or not JIRA_API_TOKEN:
            logger.warning("Confluence credentials not configured — skipping")
            return []

        url = f"{CONFLUENCE_BASE_URL}/wiki/rest/api/content/search"
        auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
        cql = (
            f'space = "{CONFLUENCE_SPACE_KEY}" AND text ~ "{query}" '
            f'ORDER BY relevance DESC'
        )
        params = {
            "cql": cql,
            "limit": top_k,
            "expand": "body.storage,metadata.labels",
        }
        resp = requests.get(url, params=params, auth=auth, timeout=15)
        resp.raise_for_status()

        results = resp.json().get("results", [])
        pages = []
        for r in results:
            body_val = (
                r.get("body", {}).get("storage", {}).get("value", "")
            )
            # Strip HTML tags for plain-text summary
            plain = re.sub(r"<[^>]+>", " ", body_val)[:1000]
            pages.append(
                {
                    "title": r.get("title", ""),
                    "url": (
                        CONFLUENCE_BASE_URL
                        + r.get("_links", {}).get("webui", "")
                    ),
                    "excerpt": plain.strip(),
                }
            )
        logger.info("Confluence returned %d pages for query '%s'", len(pages), query[:60])
        return pages
    except Exception as exc:
        logger.error("Confluence search failed: %s", exc)
        return []


def search_context(ticket: dict) -> dict:
    """Find similar tickets and Confluence articles for the given ticket.

    Args:
        ticket: dict with ``key``, ``summary``, ``description``.

    Returns:
        dict with ``similar_tickets`` (list) and ``confluence_pages`` (list).
    """
    query = f"{ticket.get('summary', '')} {ticket.get('description', '')}".strip()
    if not query:
        return {"similar_tickets": [], "confluence_pages": []}

    similar = _search_similar_tickets(query, TOP_K_SIMILAR_TICKETS)
    confluence = _search_confluence(query[:200], TOP_K_CONFLUENCE)

    logger.info(
        "Ticket %s: found %d similar tickets, %d Confluence pages",
        ticket.get("key"), len(similar), len(confluence),
    )
    return {"similar_tickets": similar, "confluence_pages": confluence}


print("Search functions defined.")

## 9. Notify Task (`tasks/notify.py`)

Sends email alerts for log thresholds and Datadog monitor breaches.

In [ ]:
"""Send email alerts for log thresholds and Datadog monitor breaches."""

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


# ---------------------------------------------------------------------------
# SMTP helper
# ---------------------------------------------------------------------------

def _send_email(subject: str, body_html: str, recipients: list[str] | None = None) -> None:
    to_list = recipients or ALERT_EMAIL_RECIPIENTS
    if not to_list:
        logger.warning("No email recipients configured — skipping email")
        return
    if not SMTP_USER or not SMTP_PASSWORD:
        logger.warning("SMTP credentials not configured — skipping email")
        return

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = SMTP_USER
    msg["To"] = ", ".join(to_list)
    msg.attach(MIMEText(body_html, "html"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.ehlo()
            server.starttls()
            server.login(SMTP_USER, SMTP_PASSWORD)
            server.sendmail(SMTP_USER, to_list, msg.as_string())
        logger.info("Email sent: %s -> %s", subject, to_list)
    except Exception as exc:
        logger.error("Failed to send email: %s", exc)
        raise


# ---------------------------------------------------------------------------
# Notification functions
# ---------------------------------------------------------------------------

def send_threshold_alert(
    ticket: dict,
    error_count: int,
    warn_count: int,
    fatal_count: int,
    top_lines: list[str],
) -> None:
    """Send an email alert when log thresholds are exceeded.

    Args:
        ticket: ticket dict (key, summary, service, env).
        error_count: number of error-level log lines.
        warn_count: number of warning-level log lines.
        fatal_count: number of fatal/critical log lines.
        top_lines: most relevant log excerpts.
    """
    ticket_key = ticket.get("key", "N/A")
    service = ticket.get("service", "unknown")
    env = ticket.get("env", "unknown")

    subject = f"\u26a0\ufe0f AI Ops Alert: Log Threshold Exceeded \u2014 {ticket_key} [{service}/{env}]"

    lines_html = "".join(
        f"<li><code>{line[:200]}</code></li>" for line in top_lines
    )

    body = f"""
    <html><body>
    <h2>\U0001f6a8 Log Threshold Alert</h2>
    <p><strong>Ticket:</strong> {ticket_key} \u2014 {ticket.get('summary', '')}</p>
    <p><strong>Service:</strong> {service} | <strong>Env:</strong> {env}</p>
    <table border="1" cellpadding="6" cellspacing="0">
      <tr><th>Metric</th><th>Count</th><th>Threshold</th></tr>
      <tr><td>Errors</td><td>{error_count}</td><td>{ERROR_COUNT_THRESHOLD}</td></tr>
      <tr><td>Warnings</td><td>{warn_count}</td><td>{WARNING_COUNT_THRESHOLD}</td></tr>
      <tr><td>Fatals</td><td>{fatal_count}</td><td>{FATAL_COUNT_THRESHOLD}</td></tr>
    </table>
    <h3>Top Log Lines</h3>
    <ul>{lines_html}</ul>
    <p>Please investigate immediately.</p>
    </body></html>
    """

    _send_email(subject, body)


def send_rca_notification(ticket: dict, rca_text: str) -> None:
    """Email the generated RCA to stakeholders.

    Args:
        ticket: ticket dict.
        rca_text: full RCA markdown text.
    """
    import html as _html

    ticket_key = ticket.get("key", "N/A")
    subject = f"\U0001f4cb AI Ops RCA Ready \u2014 {ticket_key}: {ticket.get('summary', '')[:80]}"

    rca_html = _html.escape(rca_text).replace("\n", "<br>")
    body = f"""
    <html><body>
    <h2>Root Cause Analysis \u2014 {ticket_key}</h2>
    <p><strong>Summary:</strong> {ticket.get('summary', '')}</p>
    <p><strong>Priority:</strong> {ticket.get('priority', '')} |
       <strong>Category:</strong> {ticket.get('category', '')}</p>
    <hr/>
    {rca_html}
    <hr/>
    <p><em>Generated by AI Ops Automation System</em></p>
    </body></html>
    """

    _send_email(subject, body)


def send_datadog_alert(alert_payload: dict) -> None:
    """Email stakeholders when Datadog signals a monitor threshold breach.

    Args:
        alert_payload: parsed Datadog webhook payload.
    """
    monitor_name = alert_payload.get("monitor_name", "Unknown Monitor")
    metric = alert_payload.get("metric", "")
    value = alert_payload.get("value", "")
    threshold = alert_payload.get("threshold", "")
    screenshot_url = alert_payload.get("snapshot_url", "")
    transition = alert_payload.get("transition", "ALERT")

    subject = f"\U0001f534 Datadog Monitor {transition}: {monitor_name}"

    screenshot_html = (
        f'<p><a href="{screenshot_url}">View Snapshot</a></p>'
        if screenshot_url
        else ""
    )

    body = f"""
    <html><body>
    <h2>\U0001f534 Datadog Monitor Alert</h2>
    <p><strong>Monitor:</strong> {monitor_name}</p>
    <p><strong>Metric:</strong> {metric}</p>
    <p><strong>Current Value:</strong> {value} | <strong>Threshold:</strong> {threshold}</p>
    <p><strong>Transition:</strong> {transition}</p>
    {screenshot_html}
    <pre>{str(alert_payload)[:2000]}</pre>
    <p><em>AI Ops has queued an RCA for this event.</em></p>
    </body></html>
    """

    _send_email(subject, body)


print("Notification functions defined.")

## 10. Confluence KB Task (`tasks/confluence_kb.py`)

Creates Confluence knowledge articles from AI-generated RCAs.

In [ ]:
"""Create Confluence knowledge articles from AI-generated RCAs."""


def _markdown_to_confluence_storage(md: str) -> str:
    """Very lightweight Markdown → Confluence Storage Format conversion."""
    lines = md.split("\n")
    out: list[str] = []
    for line in lines:
        # Headings
        if line.startswith("### "):
            out.append(f"<h3>{line[4:]}</h3>")
        elif line.startswith("## "):
            out.append(f"<h2>{line[3:]}</h2>")
        elif line.startswith("# "):
            out.append(f"<h1>{line[2:]}</h1>")
        # Bold
        else:
            line = re.sub(r"\*\*(.*?)\*\*", r"<strong>\1</strong>", line)
            # Bullet list
            if line.startswith("- "):
                out.append(f"<li>{line[2:]}</li>")
            else:
                out.append(f"<p>{line}</p>" if line.strip() else "<p> </p>")

    return "\n".join(out)


def create_kb_article(ticket: dict, rca_text: str) -> dict:
    """Create a new Confluence page from the RCA when no KB article exists.

    Args:
        ticket: ticket dict (key, summary, category, service).
        rca_text: full RCA text generated by the LLM.

    Returns:
        dict with ``page_id`` and ``page_url``.
    """
    import requests
    from requests.auth import HTTPBasicAuth

    if not CONFLUENCE_BASE_URL or not JIRA_API_TOKEN:
        logger.warning("Confluence credentials not configured — skipping KB creation")
        return {"page_id": None, "page_url": None}

    ticket_key = ticket.get("key", "N/A")
    summary = ticket.get("summary", "Untitled")
    category = ticket.get("category", "General")
    service = ticket.get("service", "unknown")

    title = f"[RCA] {ticket_key} \u2014 {summary[:80]}"
    storage_body = _markdown_to_confluence_storage(rca_text)

    # Wrap in a Confluence info macro header
    page_body = f"""
<ac:structured-macro ac:name="info">
  <ac:rich-text-body>
    <p>Auto-generated RCA by AI Ops for Jira ticket
       <strong>{ticket_key}</strong> ({category} / {service}).
    </p>
  </ac:rich-text-body>
</ac:structured-macro>
{storage_body}
"""

    url = f"{CONFLUENCE_BASE_URL}/wiki/rest/api/content"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)

    payload: dict = {
        "type": "page",
        "title": title,
        "space": {"key": CONFLUENCE_SPACE_KEY},
        "body": {
            "storage": {
                "value": page_body,
                "representation": "storage",
            }
        },
    }
    if CONFLUENCE_PARENT_PAGE_ID:
        payload["ancestors"] = [{"id": CONFLUENCE_PARENT_PAGE_ID}]

    resp = requests.post(url, json=payload, auth=auth, timeout=20)
    resp.raise_for_status()
    data = resp.json()
    page_id = data.get("id")
    webui = data.get("_links", {}).get("webui", "")
    page_url = CONFLUENCE_BASE_URL + webui

    logger.info("Created Confluence page '%s' (id=%s)", title, page_id)

    # Link the new Confluence page back to the Jira ticket
    _link_confluence_to_jira(ticket_key, page_url, title)

    return {"page_id": page_id, "page_url": page_url}


def _link_confluence_to_jira(ticket_key: str, page_url: str, page_title: str) -> None:
    """Add a remote link on the Jira issue pointing to the new Confluence page."""
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        return

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/remotelink"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "globalId": f"confluence-rca-{ticket_key}",
        "object": {
            "url": page_url,
            "title": page_title,
            "icon": {
                "url16x16": "https://confluence.atlassian.com/images/logo/confluence_16.png",
                "title": "Confluence Page",
            },
        },
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.warning(
            "Could not link Confluence page to Jira %s: %s", ticket_key, resp.text
        )


print("Confluence KB functions defined.")

## 11. FastAPI Application (`main.py`)

FastAPI application with Jira & Datadog webhook endpoints and pipeline orchestration.

In [ ]:
"""FastAPI application: Jira & Datadog webhook endpoints + pipeline orchestration."""

from typing import Any
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse


app = FastAPI(
    title="AI Ops Automation",
    description=(
        "End-to-end AI Ops pipeline: Jira ingestion → classification → "
        "log fetching → RCA generation → notifications."
    ),
    version="1.0.0",
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _extract_description(desc: Any) -> str:
    """Extract plain text from Jira's Atlassian Document Format description."""
    if desc is None:
        return ""
    if isinstance(desc, str):
        return desc
    # ADF: traverse content nodes
    parts: list[str] = []

    def _walk(node: Any) -> None:
        if isinstance(node, dict):
            if node.get("type") == "text":
                parts.append(node.get("text", ""))
            for child in node.get("content", []):
                _walk(child)
        elif isinstance(node, list):
            for item in node:
                _walk(item)

    _walk(desc)
    return " ".join(parts).strip()


def _extract_custom_field(fields: dict, field_id: str, default: Any) -> Any:
    val = fields.get(field_id)
    if val is None:
        return default
    if isinstance(val, dict):
        return val.get("value", default)
    return val


def _create_jira_ticket_for_datadog(payload: dict) -> str | None:
    """Create a Jira issue for a Datadog monitor alert; return the issue key."""
    try:
        import requests
        from requests.auth import HTTPBasicAuth

        if not JIRA_BASE_URL or not JIRA_API_TOKEN:
            return None

        monitor_name = payload.get("monitor_name", "Datadog Monitor Alert")
        url = f"{JIRA_BASE_URL}/rest/api/3/issue"
        auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)

        issue_payload = {
            "fields": {
                "project": {"key": JIRA_PROJECT_KEY},
                "summary": f"[Datadog Alert] {monitor_name}",
                "issuetype": {"name": "Incident"},
                "priority": {"name": "High"},
                "description": {
                    "type": "doc",
                    "version": 1,
                    "content": [
                        {
                            "type": "paragraph",
                            "content": [
                                {
                                    "type": "text",
                                    "text": (
                                        f"Automated Jira ticket created by AI Ops.\n"
                                        f"Monitor: {monitor_name}\n"
                                        f"Metric: {payload.get('metric', 'N/A')}\n"
                                        f"Value: {payload.get('value', 'N/A')}\n"
                                        f"Threshold: {payload.get('threshold', 'N/A')}"
                                    ),
                                }
                            ],
                        }
                    ],
                },
            }
        }

        resp = requests.post(url, json=issue_payload, auth=auth, timeout=15)
        if resp.ok:
            key = resp.json().get("key")
            logger.info("Created Jira ticket %s for Datadog alert", key)
            return key
        else:
            logger.error("Failed to create Jira ticket: %s %s", resp.status_code, resp.text)
            return None
    except Exception as exc:
        logger.error("Exception creating Jira ticket: %s", exc)
        return None


def _run_pipeline(ticket: dict) -> None:
    """Run the full analysis pipeline synchronously (no Celery in notebook context)."""
    # Step 1: Classify
    classify_result = classify_ticket(ticket)
    merged = {**ticket, **classify_result}

    # Step 2: Parallel context (log fetch + search)
    log_result = fetch_logs(merged)
    search_result = search_context(merged)
    context = {**log_result, **search_result}
    context["incident_risk"] = merged.get("incident_risk", 0.0)

    # Step 3: RCA
    rca_result = generate_rca(merged, context)

    # Step 4: Notify
    send_rca_notification(ticket=merged, rca_text=rca_result.get("rca_text", ""))

    return rca_result


print("FastAPI app and pipeline helpers defined.")

## Section 11: Metrics Summary


In [ ]:
# ============================================================
# Save consolidated metrics
# ============================================================
with open(f"{ART}/summary_metrics.json","w") as f:
    json.dump(summary_metrics, f, indent=2)

print("\n=== SUMMARY METRICS ===")
print(json.dumps(summary_metrics, indent=2))
print(f"\nArtifacts saved under: {ART}")

## Section 12: Manual Pipeline Trigger


## 12. Manual Pipeline Trigger (Testing)

Use this cell to manually run the full pipeline with a sample ticket — useful for testing without a live Jira webhook.

In [ ]:
# Example: manually trigger the pipeline with a sample ticket
sample_ticket = {
    "key": "OPS-123",
    "summary": "Payment service timing out for EU users",
    "description": "Users in the EU region are experiencing 504 gateway timeouts on /api/payments. Error rate spiked to 15% at 14:30 UTC.",
    "comments_text": "",
    "top_error_lines": "",
    "env": "production",
    "service": "payment-service",
    "affected_users": 5000,
    "downtime_minutes": 30,
    "error_count": 120,
    "fatal_count": 2,
    "timeout_count": 95,
    "auth_error_count": 0,
}

# Uncomment to run the full pipeline:
# result = _run_pipeline(sample_ticket)
# print(result)

# Or run individual steps:
# classify_result = classify_ticket(sample_ticket)
# print("Classification:", classify_result)

# search_result = search_context(sample_ticket)
# print("Search:", search_result)

# log_result = fetch_logs(sample_ticket)
# print("Logs:", log_result)

print("Sample ticket ready. Uncomment the lines above to run the pipeline.")